## Run Simulation


In [ ]:
# Standard library
import json
import os
import sqlite3
import subprocess
import warnings
from datetime import datetime

# Third-party
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pandas.errors import PerformanceWarning
from plotly.subplots import make_subplots

# Optional: only keep if used
import seaborn as sns  # remove if unused

# Local imports
# from plotting_module import plot_multi_simulation, plot_end_use_stack  # adjust path if needed

# Tidy warnings (example)
warnings.simplefilter(action="ignore", category=PerformanceWarning)


def get_energy_from_sql(sql_file_path, default_year, variable):
    conversion_factors = {
        'Electricity': 2.77778e-10,
        'NaturalGas': 9.48043e-9
    }

    query = """
    SELECT r.VariableValue * ? AS VariableValue, t.Month, t.Day, t.Hour, t.Minute
    FROM ReportVariableDataDictionary AS d
    JOIN ReportVariableData AS r ON d.ReportVariableDataDictionaryIndex = r.ReportVariableDataDictionaryIndex
    JOIN Time AS t ON r.TimeIndex = t.TimeIndex
    WHERE d.VariableName = ? AND d.ReportingFrequency = 'Zone Timestep'
    """

    with sqlite3.connect(sql_file_path) as conn:
        df = pd.read_sql_query(query, conn, params=(conversion_factors[variable], f"{variable}:Facility"))

    df['Year'] = default_year
    df['DateTime'] = pd.to_datetime(df[['Year', 'Month', 'Day', 'Hour', 'Minute']])
    df.set_index('DateTime', inplace=True)

    return df['VariableValue'].resample('H').sum()

def extract_simulation_data(sql_path, year):
    elec = get_energy_from_sql(sql_path, year, 'Electricity')
    gas = get_energy_from_sql(sql_path, year, 'NaturalGas')

    df_sim = pd.DataFrame({
        "Electricity_MWh": elec,
        "Gas_Therms": gas,
    })

    df_monthly = df_sim.resample('M').agg({
        "Electricity_MWh": ['sum', 'max'],
        "Gas_Therms": 'sum'
    })

    df_monthly.columns = ['Electricity_MWh', 'Max_Electricity_MW', 'Gas_Therms']
    return df_monthly.loc[str(year)]

def get_end_use_summary(sql_file_path):
    conn = sqlite3.connect(sql_file_path)
    query = """
    SELECT RowName AS EndUse, ColumnName AS FuelType, Value
    FROM TabularDataWithStrings
    WHERE ReportName = 'AnnualBuildingUtilityPerformanceSummary'
    AND TableName = 'End Uses'
    AND Value != ''
    """
    df = pd.read_sql_query(query, conn)
    conn.close()

    df['Value'] = pd.to_numeric(df['Value'], errors='coerce')
    summary = df.groupby(['EndUse', 'FuelType'])['Value'].sum().unstack(fill_value=0)
    return summary.get('Electricity', pd.Series(dtype=float)).drop(index='Total End Uses', errors='ignore')

def extract_total_ghg_emissions(sql_path, scenario_name="LRMER_MidCase_15"):
    osw_path = os.path.join(os.path.dirname(os.path.dirname(sql_path)), 'out.osw')
    try:
        with open(osw_path, "r", encoding="utf-8") as f:
            osw_data = json.load(f)
    except Exception:
        return None

    gas_ghg = None
    cambium_ghg = None
    target_key = f"Annual hourly emissions for cambium scenario '{scenario_name}' (kg CO2e)"
    alt_key = f"annual_electricity_ghg_emissions_{scenario_name}_kg"

    for step in osw_data.get("steps", []):
        for val in step.get("result", {}).get("step_values", []):
            name = val.get("name")
            if name == "annual_natural_gas_ghg_emissions_kg":
                gas_ghg = val.get("value")
            elif name == target_key or name == alt_key:
                cambium_ghg = val.get("value")

    if gas_ghg is None and cambium_ghg is None:
        return None
    return (gas_ghg or 0) + (cambium_ghg or 0)

def calculate_monthly_energy_cost(monthly_kwh, monthly_peak_kw, monthly_therms):
    base_charge = 0
    demand_rate = 0
    energy_rate = 0.09
    gas_rate = 1.33
    tax_rate = 0

    elec_cost = (
        base_charge +
        demand_rate * monthly_peak_kw +
        energy_rate * monthly_kwh
    )
    elec_cost_total = elec_cost * (1 + tax_rate)
    gas_cost_total = monthly_therms * gas_rate

    return elec_cost_total + gas_cost_total

def plot_end_use_stack(sql_paths, labels=None, output_dir=None, city_name=None):
    if labels is None:
        labels = [f"Run_{i+1}" for i in range(len(sql_paths))]

    end_use_breakdowns = [get_end_use_summary(path) for path in sql_paths]
    all_end_uses = sorted(set.union(*[set(df.index) for df in end_use_breakdowns]))

    combined_df = pd.DataFrame(index=all_end_uses, columns=labels)

    for label, df in zip(labels, end_use_breakdowns):
        combined_df[label] = df.reindex(all_end_uses).fillna(0.0)

    combined_df = combined_df[(combined_df != 0).any(axis=1)]

    def is_constant(row):
        return np.allclose(row.values, row.values[0])

    constant_rows = combined_df[combined_df.apply(is_constant, axis=1)]
    variable_rows = combined_df.drop(constant_rows.index)

    priority_top = ["Cooling", "Heating"]
    priority_rows = variable_rows.loc[variable_rows.index.intersection(priority_top)]
    other_variable_rows = variable_rows.drop(priority_rows.index)

    final_df = pd.concat([constant_rows, other_variable_rows, priority_rows])

    fig = go.Figure()

    # --- Stacked bar traces are intentionally commented out so you can re-enable them quickly if needed ---
    # for end_use in final_df.index:
    #     fig.add_trace(go.Bar(
    #         x=list(range(len(labels))),
    #         y=final_df.loc[end_use],
    #         name=end_use,
    #         width=0.4,
    #         text=[f"{v:,.0f}" if v > 0 else "" for v in final_df.loc[end_use]],
    #         textposition='inside',
    #         textfont=dict(size=20)
    #     ))

    formatted_labels = [label.replace(" - ", "<br>").replace(" (", "<br>(") for label in labels]

    title_city = f"Annual Electricity Use by End Use (Stacked) - {city_name}" if city_name else 'Annual Electricity Use by End Use (Stacked)'

    # Note: barmode='stack' is commented so the stacked plot is inactive by default.
    # To re-enable stacked bars, uncomment the loop above and the barmode line below.
    fig.update_layout(
        # barmode='stack',
        title=title_city,
        xaxis_title='Scenario',
        yaxis_title='Electricity (kWh)',
        template='simple_white',
        height=600,
        width=800,
        font=dict(size=20),
        xaxis=dict(
            tickangle=0,
            tickmode='array',
            tickvals=list(range(len(labels))),
            ticktext=formatted_labels,
            tickfont=dict(size=20),
            titlefont=dict(size=20)
        ),
        yaxis=dict(
            tickfont=dict(size=20),
            titlefont=dict(size=20)
        )
    )

    timestamp = datetime.now().strftime("%Y%m%d%H%M")
    safe_city = city_name.replace(' ', '_') if city_name else 'city'
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
        html_path = os.path.join(output_dir, f"{safe_city}_end_use_stack_{timestamp}.html")
        jpeg_path = os.path.join(output_dir, f"{safe_city}_end_use_stack_{timestamp}.jpeg")
    else:
        html_path = f"end_use_stack_{timestamp}.html"
        jpeg_path = f"end_use_stack_{timestamp}.jpeg"

    # fig.write_html(html_path)
    # try:
    #     fig.write_image(jpeg_path, format="jpeg", scale=3, width=1920, height=1440)
    # except Exception:
    #     # image export may fail if kaleido or orca isn't available; ignore but keep html
    #     pass
    # fig.show()

def plot_multi_simulation(sql_paths, year, labels_full=None, labels_short=None, output_dir=None, city_name=None):
    if labels_full is None:
        labels_full = [f"Run_{i+1}" for i in range(len(sql_paths))]
    if labels_short is None:
        labels_short = labels_full

    colors = px.colors.qualitative.Plotly
    colors = colors * ((len(sql_paths) // len(colors)) + 1)

    sim_results = [extract_simulation_data(sql, year) for sql in sql_paths]
    ghgs = [extract_total_ghg_emissions(sql) for sql in sql_paths]

    months = sim_results[0].index
    x_labels = months.strftime('%b')
    costs = []

    for df in sim_results:
        kwh = df["Electricity_MWh"] * 1000
        peak_kw = df["Max_Electricity_MW"] * 1000
        therms = df["Gas_Therms"]
        monthly_cost = calculate_monthly_energy_cost(kwh, peak_kw, therms)
        costs.append(monthly_cost)

    annual_elec = [df["Electricity_MWh"].sum() * 1000 for df in sim_results]
    annual_gas = [df["Gas_Therms"].sum() for df in sim_results]
    annual_cost = [c.sum() for c in costs]

    # Create clean short labels for subplot titles
    clean_labels = []
    for label in labels_full:
        if "2 Speed - Lab Data - Electric Backup" in label:
            clean_labels.append("NRELData-Electric")
        elif "2 Speed - Lab Data - Gas Backup" in label:
            clean_labels.append("NRELData-DualFuel")
        elif "HPC curves - Electric Backup" in label:
            clean_labels.append("HPC-Electric")
        elif "HPC curves - Gas Backup" in label or "CCHP curves - Gas Backup" in label or "HP Challenge curves - Gas Backup" in label:
            clean_labels.append("HPC-DualFuel")
        else:
            clean_labels.append(label)

    # Energy title: 1 row header + first 3 + last 2
    elec_title = "Energy (kWh)<br>" + " | ".join([f"{l}: {v:,.0f}" for l, v in zip(clean_labels[:3], annual_elec[:3])]) + "<br>" + " | ".join([f"{l}: {v:,.0f}" for l, v in zip(clean_labels[3:], annual_elec[3:])])
    
    # Gas title: 2 rows
    gas_title = "Gas (Therms)<br>" + " | ".join([f"{l}: {v:,.0f}" for l, v in zip(clean_labels[:3], annual_gas[:3])]) + "<br>" + " | ".join([f"{l}: {v:,.0f}" for l, v in zip(clean_labels[3:], annual_gas[3:])])
    
    peak_title = "Peak Electricity (kW)"
    
    # Cost title: 2 rows, in k$
    cost_k = [c / 1000 for c in annual_cost]
    cost_title = "Energy Cost<br>" + " | ".join([f"{l}: &#36;{v:,.1f}k" for l, v in zip(clean_labels[:3], cost_k[:3])]) + "<br>" + " | ".join([f"{l}: &#36;{v:,.1f}k" for l, v in zip(clean_labels[3:], cost_k[3:])])

    fig = make_subplots(
        rows=2, cols=2, shared_xaxes=False,
        subplot_titles=[elec_title, gas_title, peak_title, cost_title],
        vertical_spacing=0.35,
        horizontal_spacing=0.15
    )

    subplot_map = {
        'Electricity': (1, 1),
        'Gas': (1, 2),
        'Peak': (2, 1),
        'Cost': (2, 2)
    }

    # Create legend labels with cleaner formatting
    legend_labels = []
    for label in labels_full:
        if "2 Speed - Lab Data - Electric Backup" in label:
            legend_labels.append("2 Speed NREL Data - Electric")
        elif "2 Speed - Lab Data - Gas Backup" in label:
            legend_labels.append("2 Speed NREL Data - Dual Fuel")
        elif "HPC curves - Electric Backup" in label:
            legend_labels.append("HP Challenge - Electric")
        elif "HPC curves - Gas Backup" in label or "CCHP curves - Gas Backup" in label or "HP Challenge curves - Gas Backup" in label:
            legend_labels.append("HP Challenge - Dual Fuel")
        else:
            legend_labels.append(label)
    
    for df, legend_label, color in zip(sim_results, legend_labels, colors):
        fig.add_trace(go.Bar(x=x_labels, y=df["Electricity_MWh"] * 1000, name=legend_label, marker_color=color),
                      row=subplot_map['Electricity'][0], col=subplot_map['Electricity'][1])
        fig.add_trace(go.Bar(x=x_labels, y=df["Gas_Therms"], name=legend_label, marker_color=color, showlegend=False),
                      row=subplot_map['Gas'][0], col=subplot_map['Gas'][1])
        fig.add_trace(go.Bar(x=x_labels, y=df["Max_Electricity_MW"] * 1000, name=legend_label, marker_color=color, showlegend=False),
                      row=subplot_map['Peak'][0], col=subplot_map['Peak'][1])

    for cost, legend_label, color in zip(costs, legend_labels, colors):
        fig.add_trace(go.Bar(x=x_labels, y=cost, name=legend_label, marker_color=color, showlegend=False),
                      row=subplot_map['Cost'][0], col=subplot_map['Cost'][1])

    # Include city name in the main title if provided
    main_title = f"Simulation Comparison - {city_name}" if city_name else "Simulation Comparison"

    fig.update_layout(
    height=800,
    width=960,
    title=dict(
        text=main_title,
        y=0.98,
        x=0.5,
        xanchor='center',
        yanchor='top',
        font=dict(size=16)
    ),
    template='simple_white',
    barmode='group',
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="top",
        y=0.50,
        xanchor="center",
        x=0.5,
        font=dict(size=11),
        bgcolor="rgba(255,255,255,0.9)",
        bordercolor="gray",
        borderwidth=1
    ),
    margin=dict(t=80)
    )

    for i in range(4):
        fig.layout.annotations[i].font.size = 10

    timestamp = pd.Timestamp.now().strftime("%Y%m%d%H%M")
    safe_city = city_name.replace(' ', '_') if city_name else 'city'
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
        html_path = os.path.join(output_dir, f"bar_plots/{safe_city}_comparison_scenarios.html")
        jpeg_path = os.path.join(output_dir, f"bar_plots/{safe_city}_comparison_scenarios.jpeg")
    else:
        html_path = f"comparison_plus_ghg_{year}_{timestamp}.html"
        jpeg_path = f"comparison_plus_ghg_{year}_{timestamp}.jpeg"

    # fig.write_html(html_path)
    try:
        fig.write_image(jpeg_path, format="jpeg", scale=2, width=960, height=720)
    except Exception:
        # image export may fail if kaleido/orca isn't available; ignore but keep html
        pass
    fig.show()

def get_city_weather_files(city_name):
    """
    Get the EPW and DDY file paths for a given city name
    Args:
        city_name (str): City name from the city_region_state dictionary
    Returns:
        dict: {'epw': <path or None>, 'ddy': <path or None>}
    """
    if city_name not in city_region_state:   # <-- was city_climate_zone
        print(f"Error: '{city_name}' not found in city_region_state dictionary")
        return None
    folder_name = 'USA_PA_Lancaster.AP.725116_TMYx' if city_name == 'Lancaster_TMYx' else city_name
    city_folder_path = os.path.join(base_weather_path, folder_name)
    if not os.path.exists(city_folder_path):
        print(f"Error: Folder '{city_folder_path}' does not exist")
        return None
    epw_file = ddy_file = None
    for filename in os.listdir(city_folder_path):
        if filename.lower().endswith('.epw'):
            epw_file = os.path.join(city_folder_path, filename)
        elif filename.lower().endswith('.ddy'):
            ddy_file = os.path.join(city_folder_path, filename)
    return {'epw': epw_file, 'ddy': ddy_file}

def create_city_scenarios(city, base_run_dir, measure_dir_path, seed_model_path, overwrite_existing=False):
    # Weather
    wf = get_city_weather_files(city)
    if wf is None or wf['epw'] is None:
        print(f"❌ Weather files not found for {city}")
        return
    epw_path, ddy_path = wf['epw'], wf['ddy']

    # Lookups for this city
    info = city_region_state[city]
    grid_state = info['state']              # e.g., 'CO'
    grid_region = info['cambium_region']    # e.g., 'RMPAc'

    # Your scenarios (unchanged)
    scenarios = [
        {
            "name": "Baseline",
            "upgrade_args": {
                "hprtu_scenario": "Baseline",
                "backup_ht_fuel_scheme": "electric_resistance_backup",
                "performance_oversizing_factor": 0,
                "htg_sizing_option": "0F",
                "clg_oversizing_estimate": 1,
                "htg_to_clg_hp_ratio": 1.0,
                "hp_min_comp_lockout_temp_f": 0.0,
                "hr": False, "dcv": False, "econ": False,
                "sizing_run": False, "debug_verbose": False
            },
        },
        {
            "name": "2 Speed - Lab Data - Electric Backup",
            "upgrade_args": {
                "hprtu_scenario": "two_speed_lab_data",
                "backup_ht_fuel_scheme": "electric_resistance_backup",
                "performance_oversizing_factor": 0,
                "htg_sizing_option": "0F",
                "clg_oversizing_estimate": 1,
                "htg_to_clg_hp_ratio": 1.0,
                "hp_min_comp_lockout_temp_f": 0.0,
                "hr": False, "dcv": False, "econ": False,
                "sizing_run": False, "debug_verbose": False
            },
        },
        {
            "name": "2 Speed - Lab Data - Gas Backup",
            "upgrade_args": {
                "hprtu_scenario": "two_speed_lab_data",
                "backup_ht_fuel_scheme": "match_original_primary_heating_fuel",
                "performance_oversizing_factor": 0,
                "htg_sizing_option": "0F",
                "clg_oversizing_estimate": 1,
                "htg_to_clg_hp_ratio": 1.0,
                "hp_min_comp_lockout_temp_f": 30.0,
                "hr": False, "dcv": False, "econ": False,
                "sizing_run": False, "debug_verbose": False
            },
        },
        {
            "name": "HPC curves - Electric Backup",
            "upgrade_args": {
                "hprtu_scenario": "cchpc_2027_spec",
                "backup_ht_fuel_scheme": "electric_resistance_backup",
                "performance_oversizing_factor": 0,
                "htg_sizing_option": "0F",
                "clg_oversizing_estimate": 1,
                "htg_to_clg_hp_ratio": 1.0,
                "hp_min_comp_lockout_temp_f": -10.0,
                "hr": False, "dcv": False, "econ": False,
                "sizing_run": False, "debug_verbose": False
            },
        },
        {
            "name": "HP Challenge curves - Gas Backup",
            "upgrade_args": {
                "hprtu_scenario": "cchpc_2027_spec",
                "backup_ht_fuel_scheme": "match_original_primary_heating_fuel",
                "performance_oversizing_factor": 0,
                "htg_sizing_option": "0F",
                "clg_oversizing_estimate": 1,
                "htg_to_clg_hp_ratio": 1.0,
                "hp_min_comp_lockout_temp_f": -10.0,
                "hr": False, "dcv": False, "econ": False,
                "sizing_run": False, "debug_verbose": False
            },
        }
    ]

    city_run_dir = os.path.join(base_run_dir, city)
    os.makedirs(city_run_dir, exist_ok=True)

    for scenario in scenarios:
        run_dir = os.path.join(city_run_dir, scenario["name"])
        sql_output_path = os.path.join(run_dir, "run", "eplusout.sql")

        # Skip/overwrite logic
        if os.path.exists(sql_output_path) and not overwrite_existing:
            print(f"⏭️  Skipping {scenario['name']} - simulation already exists at {sql_output_path}")
            continue
        elif os.path.exists(sql_output_path) and overwrite_existing:
            print(f"♻️  Overwriting existing simulation for {scenario['name']}")

        os.makedirs(run_dir, exist_ok=True)
        osw_path = os.path.join(run_dir, "in.osw")

        # 1) Model upgrade measure (your existing one)
        steps = [{
            "measure_dir_name": "target_upgrade_measure",
            "arguments": scenario["upgrade_args"]
        }]

        # 2) Emissions reporting measure (added)
        steps.append({
            "measure_dir_name": EMISSIONS_MEASURE_DIR_NAME,
            "arguments": {
                "grid_region": grid_region,                # e.g., 'RMPAc'
                "grid_state":  grid_state,                 # e.g., 'CO'
                "emissions_scenario": EMISSIONS_SCENARIO   # e.g., 'LRMER_MidCase_15'
            }
        })

        osw = {
            "seed_file": seed_model_path,
            "weather_file": epw_path,
            "measure_paths": [measure_dir_path],
            "steps": steps,
            "name": scenario["name"]
        }

        with open(osw_path, "w") as f:
            json.dump(osw, f, indent=2)
        print(f"📝 Created OSW for {scenario['name']} at {osw_path}")
        try:
            result = subprocess.run(["openstudio", "run", "-w", osw_path], check=True, capture_output=True, text=True)
            print(f"✅ {scenario['name']} simulation completed successfully.\n")
            print(result.stdout)
        except subprocess.CalledProcessError as e:
            print(f"❌ {scenario['name']} simulation failed.")
            print("STDOUT:\n", e.stdout)
            print("STDERR:\n", e.stderr)


# === CITY MAP (state, climate zone, cambium region) ===
city_region_state = {
    'Miami':        {'state': 'FL', 'climate_zone': '1A', 'cambium_region': 'FRCCc'},
    'Houston':      {'state': 'TX', 'climate_zone': '2A', 'cambium_region': 'ERCTc'},
    'Phoenix':      {'state': 'AZ', 'climate_zone': '2B', 'cambium_region': 'AZNMc'},

    'Atlanta':      {'state': 'GA', 'climate_zone': '3A', 'cambium_region': 'SRSOc'},
    'ElPaso':       {'state': 'TX', 'climate_zone': '3B', 'cambium_region': 'AZNMc'},  # WECC/AZ-NM split
    'SanFrancisco': {'state': 'CA', 'climate_zone': '3C', 'cambium_region': 'CAMXc'},

    'Lancaster':    {'state': 'PA', 'climate_zone': '4A', 'cambium_region': 'RFCEc'},  # PA variant
    'Amarillo':     {'state': 'TX', 'climate_zone': '4B', 'cambium_region': 'SPSOc'},
    'Portland':     {'state': 'OR', 'climate_zone': '4C', 'cambium_region': 'NWPPc'},

    'Chicago':      {'state': 'IL', 'climate_zone': '5A', 'cambium_region': 'RFCWc'},
    'Denver':       {'state': 'CO', 'climate_zone': '5B', 'cambium_region': 'RMPAc'},
    'PortAngeles':  {'state': 'WA', 'climate_zone': '5C', 'cambium_region': 'NWPPc'},

    'Minneapolis':  {'state': 'MN', 'climate_zone': '6A', 'cambium_region': 'MROEc'},
    'Helena':       {'state': 'MT', 'climate_zone': '6B', 'cambium_region': 'NWPPc'},

    'Duluth':       {'state': 'MN', 'climate_zone': '7',  'cambium_region': 'MROEc'},
    'Fairbanks':    {'state': 'AK', 'climate_zone': '8',  'cambium_region': 'AKGD'},
}

city_scenario_emissions = \
    {'Amarillo': {'cambium_region': 'SPSOc',
                'climate_zone': '4B',
                'scenarios': {'2 Speed - Lab Data - Electric Backup': {'electric_ghg_kg': 351526.70522070565,
                                                                        'gas_ghg_kg': 0.0,
                                                                        'total_ghg_kg': 351526.70522070565},
                                '2 Speed - Lab Data - Gas Backup': {'electric_ghg_kg': 331004.19305434334,
                                                                    'gas_ghg_kg': 38172.403212890305,
                                                                    'total_ghg_kg': 369176.59626723366},
                                'Baseline': {'electric_ghg_kg': 315393.1602415128,
                                            'gas_ghg_kg': 107020.95592279601,
                                            'total_ghg_kg': 422414.11616430886},
                                'HP Challenge curves - Gas Backup': {'electric_ghg_kg': 326787.75970801804,
                                                                    'gas_ghg_kg': 9367.744898677971,
                                                                    'total_ghg_kg': 336155.504606696},
                                'HPC curves - Electric Backup': {'electric_ghg_kg': 334430.77425757,
                                                                'gas_ghg_kg': 0.0,
                                                                'total_ghg_kg': 334430.77425757}},
                'state': 'TX'},
    'Atlanta': {'cambium_region': 'SRSOc',
                'climate_zone': '3A',
                'scenarios': {'2 Speed - Lab Data - Electric Backup': {'electric_ghg_kg': 696888.6980586782,
                                                                        'gas_ghg_kg': 0.0,
                                                                        'total_ghg_kg': 696888.6980586782},
                            '2 Speed - Lab Data - Gas Backup': {'electric_ghg_kg': 677081.2167555644,
                                                                'gas_ghg_kg': 15437.253220182769,
                                                                'total_ghg_kg': 692518.4699757472},
                            'Baseline': {'electric_ghg_kg': 644730.4711160931,
                                            'gas_ghg_kg': 73794.81365638413,
                                            'total_ghg_kg': 718525.2847724772},
                            'HP Challenge curves - Gas Backup': {'electric_ghg_kg': 653270.2308693863,
                                                                    'gas_ghg_kg': 5887.134719522604,
                                                                    'total_ghg_kg': 659157.3655889089},
                            'HPC curves - Electric Backup': {'electric_ghg_kg': 664075.3066122659,
                                                                'gas_ghg_kg': 0.0,
                                                                'total_ghg_kg': 664075.3066122659}},
                'state': 'GA'},
    'Chicago': {'cambium_region': 'RFCWc',
                'climate_zone': '5A',
                'scenarios': {'2 Speed - Lab Data - Electric Backup': {'electric_ghg_kg': 778018.8706470928,
                                                                        'gas_ghg_kg': 0.0,
                                                                        'total_ghg_kg': 778018.8706470928},
                            '2 Speed - Lab Data - Gas Backup': {'electric_ghg_kg': 629721.5356646575,
                                                                'gas_ghg_kg': 135540.86457028912,
                                                                'total_ghg_kg': 765262.4002349466},
                            'Baseline': {'electric_ghg_kg': 586344.9359811083,
                                            'gas_ghg_kg': 227486.2909723954,
                                            'total_ghg_kg': 813831.2269535037},
                            'HP Challenge curves - Gas Backup': {'electric_ghg_kg': 673334.1008380228,
                                                                    'gas_ghg_kg': 26570.81161525765,
                                                                    'total_ghg_kg': 699904.9124532804},
                            'HPC curves - Electric Backup': {'electric_ghg_kg': 717906.5435022695,
                                                                'gas_ghg_kg': 0.0,
                                                                'total_ghg_kg': 717906.5435022695}},
                'state': 'IL'},
    'Denver': {'cambium_region': 'RMPAc',
                'climate_zone': '5B',
                'scenarios': {'2 Speed - Lab Data - Electric Backup': {'electric_ghg_kg': 346614.2653929,
                                                                    'gas_ghg_kg': 0.0,
                                                                    'total_ghg_kg': 346614.2653929},
                            '2 Speed - Lab Data - Gas Backup': {'electric_ghg_kg': 293454.2409007319,
                                                                'gas_ghg_kg': 84403.7864080301,
                                                                'total_ghg_kg': 377858.02730876196},
                            'Baseline': {'electric_ghg_kg': 279891.0916101744,
                                        'gas_ghg_kg': 150327.29386556367,
                                        'total_ghg_kg': 430218.3854757381},
                            'HP Challenge curves - Gas Backup': {'electric_ghg_kg': 302605.5433134357,
                                                                'gas_ghg_kg': 23130.957257802755,
                                                                'total_ghg_kg': 325736.50057123846},
                            'HPC curves - Electric Backup': {'electric_ghg_kg': 322640.54799793015,
                                                            'gas_ghg_kg': 0.0,
                                                            'total_ghg_kg': 322640.54799793015}},
                'state': 'CO'},
    'Duluth': {'cambium_region': 'MROEc',
                'climate_zone': '7',
                'scenarios': {'2 Speed - Lab Data - Electric Backup': {'electric_ghg_kg': 591075.7946019087,
                                                                    'gas_ghg_kg': 0.0,
                                                                    'total_ghg_kg': 591075.7946019087},
                            '2 Speed - Lab Data - Gas Backup': {'electric_ghg_kg': 401047.36726700235,
                                                                'gas_ghg_kg': 235478.8861375336,
                                                                'total_ghg_kg': 636526.2534045359},
                            'Baseline': {'electric_ghg_kg': 377686.6921479297,
                                        'gas_ghg_kg': 318208.6487726481,
                                        'total_ghg_kg': 695895.3409205778},
                            'HP Challenge curves - Gas Backup': {'electric_ghg_kg': 466909.13238502544,
                                                                'gas_ghg_kg': 60160.78290832244,
                                                                'total_ghg_kg': 527069.9152933479},
                            'HPC curves - Electric Backup': {'electric_ghg_kg': 531988.8908615072,
                                                            'gas_ghg_kg': 0.0,
                                                            'total_ghg_kg': 531988.8908615072}},
                'state': 'MN'},
    'ElPaso': {'cambium_region': 'AZNMc',
                'climate_zone': '3B',
                'scenarios': {'2 Speed - Lab Data - Electric Backup': {'electric_ghg_kg': 316888.89238194376,
                                                                    'gas_ghg_kg': 0.0,
                                                                    'total_ghg_kg': 316888.89238194376},
                            '2 Speed - Lab Data - Gas Backup': {'electric_ghg_kg': 308692.0987442496,
                                                                'gas_ghg_kg': 11456.19769660172,
                                                                'total_ghg_kg': 320148.29644085135},
                            'Baseline': {'electric_ghg_kg': 297809.34762585466,
                                        'gas_ghg_kg': 56468.1921588145,
                                        'total_ghg_kg': 354277.5397846692},
                            'HP Challenge curves - Gas Backup': {'electric_ghg_kg': 303475.13852176466,
                                                                'gas_ghg_kg': 3795.654129057804,
                                                                'total_ghg_kg': 307270.7926508225},
                            'HPC curves - Electric Backup': {'electric_ghg_kg': 306938.7516625837,
                                                            'gas_ghg_kg': 0.0,
                                                            'total_ghg_kg': 306938.7516625837}},
                'state': 'TX'},
    'Fairbanks': {'cambium_region': 'AKGD',
                'climate_zone': '8',
                'scenarios': {'2 Speed - Lab Data - Electric Backup': {'electric_ghg_kg': 1288231.258453032,
                                                                        'gas_ghg_kg': 0.0,
                                                                        'total_ghg_kg': 1288231.258453032},
                                '2 Speed - Lab Data - Gas Backup': {'electric_ghg_kg': 687510.6820353497,
                                                                    'gas_ghg_kg': 442119.8645294454,
                                                                    'total_ghg_kg': 1129630.546564795},
                                'Baseline': {'electric_ghg_kg': 664253.1215246429,
                                            'gas_ghg_kg': 493630.12440187647,
                                            'total_ghg_kg': 1157883.2459265194},
                                'HP Challenge curves - Gas Backup': {'electric_ghg_kg': 799903.009454943,
                                                                    'gas_ghg_kg': 253543.27748737423,
                                                                    'total_ghg_kg': 1053446.2869423172},
                                'HPC curves - Electric Backup': {'electric_ghg_kg': 1189468.5163717852,
                                                                'gas_ghg_kg': 0.0,
                                                                'total_ghg_kg': 1189468.5163717852}},
                'state': 'AK'},
    'Helena': {'cambium_region': 'NWPPc',
                'climate_zone': '6B',
                'scenarios': {'2 Speed - Lab Data - Electric Backup': {'electric_ghg_kg': 228140.1992385928,
                                                                    'gas_ghg_kg': 0.0,
                                                                    'total_ghg_kg': 228140.1992385928},
                            '2 Speed - Lab Data - Gas Backup': {'electric_ghg_kg': 174694.16498302997,
                                                                'gas_ghg_kg': 141947.84712758573,
                                                                'total_ghg_kg': 316642.01211061573},
                            'Baseline': {'electric_ghg_kg': 162501.79799334498,
                                        'gas_ghg_kg': 241795.81197279677,
                                        'total_ghg_kg': 404297.60996614175},
                            'HP Challenge curves - Gas Backup': {'electric_ghg_kg': 186774.26415215884,
                                                                'gas_ghg_kg': 43584.55773904044,
                                                                'total_ghg_kg': 230358.82189119927},
                            'HPC curves - Electric Backup': {'electric_ghg_kg': 208562.46723831922,
                                                            'gas_ghg_kg': 0.0,
                                                            'total_ghg_kg': 208562.46723831922}},
                'state': 'MT'},
    'Houston': {'cambium_region': 'ERCTc',
                'climate_zone': '2A',
                'scenarios': {'2 Speed - Lab Data - Electric Backup': {'electric_ghg_kg': 168077.9089757077,
                                                                        'gas_ghg_kg': 0.0,
                                                                        'total_ghg_kg': 168077.9089757077},
                            '2 Speed - Lab Data - Gas Backup': {'electric_ghg_kg': 167562.9271379138,
                                                                'gas_ghg_kg': 2184.487076549096,
                                                                'total_ghg_kg': 169747.4142144629},
                            'Baseline': {'electric_ghg_kg': 164901.6933141651,
                                            'gas_ghg_kg': 32520.576609187938,
                                            'total_ghg_kg': 197422.26992335304},
                            'HP Challenge curves - Gas Backup': {'electric_ghg_kg': 158077.47350275257,
                                                                    'gas_ghg_kg': 1738.695376158028,
                                                                    'total_ghg_kg': 159816.1688789106},
                            'HPC curves - Electric Backup': {'electric_ghg_kg': 158528.56512211042,
                                                                'gas_ghg_kg': 0.0,
                                                                'total_ghg_kg': 158528.56512211042}},
                'state': 'TX'},
    'Lancaster': {'cambium_region': 'RFCEc',
                'climate_zone': '4A',
                'scenarios': {'2 Speed - Lab Data - Electric Backup': {'electric_ghg_kg': 657739.8655027439,
                                                                        'gas_ghg_kg': 0.0,
                                                                        'total_ghg_kg': 657739.8655027439},
                                '2 Speed - Lab Data - Gas Backup': {'electric_ghg_kg': 593807.819997719,
                                                                    'gas_ghg_kg': 66935.25473457968,
                                                                    'total_ghg_kg': 660743.0747322987},
                                'Baseline': {'electric_ghg_kg': 550437.3829095861,
                                            'gas_ghg_kg': 163119.1732624039,
                                            'total_ghg_kg': 713556.55617199},
                                'HP Challenge curves - Gas Backup': {'electric_ghg_kg': 597715.8354562817,
                                                                    'gas_ghg_kg': 15577.187546296935,
                                                                    'total_ghg_kg': 613293.0230025786},
                                'HPC curves - Electric Backup': {'electric_ghg_kg': 622532.531044547,
                                                                'gas_ghg_kg': 0.0,
                                                                'total_ghg_kg': 622532.531044547}},
                'state': 'PA'},
    'Miami': {'cambium_region': 'FRCCc',
            'climate_zone': '1A',
            'scenarios': {'2 Speed - Lab Data - Electric Backup': {'electric_ghg_kg': 594702.9680514837,
                                                                    'gas_ghg_kg': 0.0,
                                                                    'total_ghg_kg': 594702.9680514837},
                            '2 Speed - Lab Data - Gas Backup': {'electric_ghg_kg': 594653.42172567,
                                                                'gas_ghg_kg': 42.45347106380536,
                                                                'total_ghg_kg': 594695.8751967337},
                            'Baseline': {'electric_ghg_kg': 592411.1259619173,
                                        'gas_ghg_kg': 2632.7044759975447,
                                        'total_ghg_kg': 595043.8304379149},
                            'HP Challenge curves - Gas Backup': {'electric_ghg_kg': 555220.057112554,
                                                                'gas_ghg_kg': 43.07969880321136,
                                                                'total_ghg_kg': 555263.1368113572},
                            'HPC curves - Electric Backup': {'electric_ghg_kg': 555270.5503370748,
                                                            'gas_ghg_kg': 0.0,
                                                            'total_ghg_kg': 555270.5503370748}},
            'state': 'FL'},
    'Minneapolis': {'cambium_region': 'MROEc',
                    'climate_zone': '6A',
                    'scenarios': {'2 Speed - Lab Data - Electric Backup': {'electric_ghg_kg': 542414.8054643566,
                                                                            'gas_ghg_kg': 0.0,
                                                                            'total_ghg_kg': 542414.8054643566},
                                '2 Speed - Lab Data - Gas Backup': {'electric_ghg_kg': 417830.8546583154,
                                                                    'gas_ghg_kg': 160379.75783159895,
                                                                    'total_ghg_kg': 578210.6124899144},
                                'Baseline': {'electric_ghg_kg': 389958.83626306563,
                                                'gas_ghg_kg': 249825.9709303828,
                                                'total_ghg_kg': 639784.8071934484},
                                'HP Challenge curves - Gas Backup': {'electric_ghg_kg': 458969.0126917693,
                                                                        'gas_ghg_kg': 32654.837175009357,
                                                                        'total_ghg_kg': 491623.84986677865},
                                'HPC curves - Electric Backup': {'electric_ghg_kg': 495380.6111847591,
                                                                    'gas_ghg_kg': 0.0,
                                                                    'total_ghg_kg': 495380.6111847591}},
                    'state': 'MN'},
    'Phoenix': {'cambium_region': 'AZNMc',
                'climate_zone': '2B',
                'scenarios': {'2 Speed - Lab Data - Electric Backup': {'electric_ghg_kg': 337363.36863567703,
                                                                        'gas_ghg_kg': 0.0,
                                                                        'total_ghg_kg': 337363.36863567703},
                            '2 Speed - Lab Data - Gas Backup': {'electric_ghg_kg': 336453.83025730075,
                                                                'gas_ghg_kg': 984.3888477207063,
                                                                'total_ghg_kg': 337438.21910502145},
                            'Baseline': {'electric_ghg_kg': 329717.954630491,
                                            'gas_ghg_kg': 25049.378019706397,
                                            'total_ghg_kg': 354767.3326501974},
                            'HP Challenge curves - Gas Backup': {'electric_ghg_kg': 325475.0366742391,
                                                                    'gas_ghg_kg': 962.4294734860158,
                                                                    'total_ghg_kg': 326437.4661477251},
                            'HPC curves - Electric Backup': {'electric_ghg_kg': 326362.91863877163,
                                                                'gas_ghg_kg': 0.0,
                                                                'total_ghg_kg': 326362.91863877163}},
                'state': 'AZ'},
    'PortAngeles': {'cambium_region': 'NWPPc',
                    'climate_zone': '5C',
                    'scenarios': {'2 Speed - Lab Data - Electric Backup': {'electric_ghg_kg': 177943.32961700758,
                                                                            'gas_ghg_kg': 0.0,
                                                                            'total_ghg_kg': 177943.32961700758},
                                '2 Speed - Lab Data - Gas Backup': {'electric_ghg_kg': 161750.27114975665,
                                                                    'gas_ghg_kg': 34703.42907826,
                                                                    'total_ghg_kg': 196453.70022801665},
                                'Baseline': {'electric_ghg_kg': None,
                                                'gas_ghg_kg': None,
                                                'total_ghg_kg': None},
                                'HP Challenge curves - Gas Backup': {'electric_ghg_kg': None,
                                                                        'gas_ghg_kg': None,
                                                                        'total_ghg_kg': None},
                                'HPC curves - Electric Backup': {'electric_ghg_kg': None,
                                                                    'gas_ghg_kg': None,
                                                                    'total_ghg_kg': None}},
                    'state': 'WA'},
    'Portland': {'cambium_region': 'NWPPc',
                'climate_zone': '4C',
                'scenarios': {'2 Speed - Lab Data - Electric Backup': {'electric_ghg_kg': 217026.35722973064,
                                                                        'gas_ghg_kg': 0.0,
                                                                        'total_ghg_kg': 217026.35722973064},
                                '2 Speed - Lab Data - Gas Backup': {'electric_ghg_kg': 176003.5318338822,
                                                                    'gas_ghg_kg': 120145.22282838792,
                                                                    'total_ghg_kg': 296148.75466227013},
                                'Baseline': {'electric_ghg_kg': 161720.51232643198,
                                            'gas_ghg_kg': 228900.63308339607,
                                            'total_ghg_kg': 390621.14540982805},
                                'HP Challenge curves - Gas Backup': {'electric_ghg_kg': 188270.59903883352,
                                                                    'gas_ghg_kg': 26679.995535843093,
                                                                    'total_ghg_kg': 214950.59457467662},
                                'HPC curves - Electric Backup': {'electric_ghg_kg': 201469.22971471242,
                                                                'gas_ghg_kg': 0.0,
                                                                'total_ghg_kg': 201469.22971471242}},
                'state': 'OR'},
    'SanFrancisco': {'cambium_region': 'CAMXc',
                    'climate_zone': '3C',
                    'scenarios': {'2 Speed - Lab Data - Electric Backup': {'electric_ghg_kg': 49590.5225294549,
                                                                            'gas_ghg_kg': 0.0,
                                                                            'total_ghg_kg': 49590.5225294549},
                                    '2 Speed - Lab Data - Gas Backup': {'electric_ghg_kg': 48820.99700086938,
                                                                        'gas_ghg_kg': 3986.4812603862474,
                                                                        'total_ghg_kg': 52807.47826125562},
                                    'Baseline': {'electric_ghg_kg': 46550.683243282016,
                                                'gas_ghg_kg': 51984.18943800643,
                                                'total_ghg_kg': 98534.87268128846},
                                    'HP Challenge curves - Gas Backup': {'electric_ghg_kg': 48097.30315133117,
                                                                        'gas_ghg_kg': 3403.308628786101,
                                                                        'total_ghg_kg': 51500.61178011727},
                                    'HPC curves - Electric Backup': {'electric_ghg_kg': 48707.559597554144,
                                                                    'gas_ghg_kg': 0.0,
                                                                    'total_ghg_kg': 48707.559597554144}},
                    'state': 'CA'}}



# === FIXED PATHS ===
seed_model_path = '/Users/cbianchi/Documents/GitHub/Target_HPs/Modeling_resources/Target_BASELINE_seed.osm'
base_weather_path = '/Users/cbianchi/Documents/GitHub/Target_HPs/Modeling_resources/weather'
measure_dir_path = '/Users/cbianchi/Documents/GitHub/Target_HPs/Modeling_resources/measures'
base_run_dir = '/Users/cbianchi/Documents/GitHub/Target_HPs/Enterprise/simulation_run'
# Which Cambium scenario to compute in the reporting measure (matches your extractor)
EMISSIONS_SCENARIO = 'LRMER_MidCase_30'

# Folder name of the reporting measure inside measure_dir_path
EMISSIONS_MEASURE_DIR_NAME = 'emissions_reporting'


PLOT_ONLY = True
OVERWRITE_EXISTING = False

RUN_SIMULATIONS_PLOTS = False

if RUN_SIMULATIONS_PLOTS:
    for city, info in city_region_state.items():
        if not PLOT_ONLY:
            create_city_scenarios(city, base_run_dir, measure_dir_path, seed_model_path, overwrite_existing=OVERWRITE_EXISTING)
        else:
            print(f"📌 PLOT_ONLY=True: skipping simulation runs for {city} and only collecting existing SQL outputs.")

        city_run_dir = os.path.join(base_run_dir, city)
        plots_dir = os.path.join(os.path.dirname(base_run_dir), 'plots')
        os.makedirs(plots_dir, exist_ok=True)

        sql_paths, labels_full, labels_short = [], [], []
        for name in [
            "Baseline",
            "2 Speed - Lab Data - Electric Backup",
            "2 Speed - Lab Data - Gas Backup",
            "HPC curves - Electric Backup",
            "HP Challenge curves - Gas Backup"
        ]:
            scenario_dir = os.path.join(city_run_dir, name)
            sql_path = os.path.join(scenario_dir, "run", "eplusout.sql")
            if os.path.exists(sql_path):
                sql_paths.append(sql_path)
                labels_full.append(name)
                labels_short.append(name.split(' ')[0])
            else:
                print(f"⚠️ SQL output not found for {name} at expected path: {sql_path}")

        if sql_paths:
            city_with_zone = f"{city} (Climate Zone {info['climate_zone']})"
            try:
                plot_multi_simulation(sql_paths, YEAR if 'YEAR' in globals() else 2020,
                                    labels_full, labels_short, output_dir=plots_dir, city_name=city_with_zone)
                # plot_end_use_stack(sql_paths, labels_full, output_dir=plots_dir, city_name=city_with_zone)
                print(f"📈 Plots written to {plots_dir} for city {city}")
            except Exception as e:
                print(f"❌ Failed to generate plots for {city}: {e}")
        else:
            print(f"No SQL results found for city {city}; skipping plots.")



## PortAngeles


In [15]:
# Standard library
import json
import os
import sqlite3
import subprocess
import warnings
from datetime import datetime

# Third-party
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pandas.errors import PerformanceWarning
from plotly.subplots import make_subplots

# Optional: only keep if used
import seaborn as sns  # remove if unused

# Tidy warnings
warnings.simplefilter(action="ignore", category=PerformanceWarning)


def get_energy_from_sql(sql_file_path, default_year, variable):
    conversion_factors = {
        'Electricity': 2.77778e-10,
        'NaturalGas': 9.48043e-9
    }

    query = """
    SELECT r.VariableValue * ? AS VariableValue, t.Month, t.Day, t.Hour, t.Minute
    FROM ReportVariableDataDictionary AS d
    JOIN ReportVariableData AS r ON d.ReportVariableDataDictionaryIndex = r.ReportVariableDataDictionaryIndex
    JOIN Time AS t ON r.TimeIndex = t.TimeIndex
    WHERE d.VariableName = ? AND d.ReportingFrequency = 'Zone Timestep'
    """

    with sqlite3.connect(sql_file_path) as conn:
        df = pd.read_sql_query(query, conn, params=(conversion_factors[variable], f"{variable}:Facility"))

    df['Year'] = default_year
    df['DateTime'] = pd.to_datetime(df[['Year', 'Month', 'Day', 'Hour', 'Minute']])
    df.set_index('DateTime', inplace=True)

    return df['VariableValue'].resample('H').sum()

def extract_simulation_data(sql_path, year):
    elec = get_energy_from_sql(sql_path, year, 'Electricity')
    gas = get_energy_from_sql(sql_path, year, 'NaturalGas')

    df_sim = pd.DataFrame({
        "Electricity_MWh": elec,
        "Gas_Therms": gas,
    })

    df_monthly = df_sim.resample('M').agg({
        "Electricity_MWh": ['sum', 'max'],
        "Gas_Therms": 'sum'
    })

    df_monthly.columns = ['Electricity_MWh', 'Max_Electricity_MW', 'Gas_Therms']
    return df_monthly.loc[str(year)]

def get_end_use_summary(sql_file_path):
    conn = sqlite3.connect(sql_file_path)
    query = """
    SELECT RowName AS EndUse, ColumnName AS FuelType, Value
    FROM TabularDataWithStrings
    WHERE ReportName = 'AnnualBuildingUtilityPerformanceSummary'
    AND TableName = 'End Uses'
    AND Value != ''
    """
    df = pd.read_sql_query(query, conn)
    conn.close()

    df['Value'] = pd.to_numeric(df['Value'], errors='coerce')
    summary = df.groupby(['EndUse', 'FuelType'])['Value'].sum().unstack(fill_value=0)
    return summary.get('Electricity', pd.Series(dtype=float)).drop(index='Total End Uses', errors='ignore')

def extract_total_ghg_emissions(sql_path, scenario_name="LRMER_MidCase_15"):
    osw_path = os.path.join(os.path.dirname(os.path.dirname(sql_path)), 'out.osw')
    try:
        with open(osw_path, "r", encoding="utf-8") as f:
            osw_data = json.load(f)
    except Exception:
        return None

    gas_ghg = None
    cambium_ghg = None
    target_key = f"Annual hourly emissions for cambium scenario '{scenario_name}' (kg CO2e)"
    alt_key = f"annual_electricity_ghg_emissions_{scenario_name}_kg"

    for step in osw_data.get("steps", []):
        for val in step.get("result", {}).get("step_values", []):
            name = val.get("name")
            if name == "annual_natural_gas_ghg_emissions_kg":
                gas_ghg = val.get("value")
            elif name == target_key or name == alt_key:
                cambium_ghg = val.get("value")

    if gas_ghg is None and cambium_ghg is None:
        return None
    return (gas_ghg or 0) + (cambium_ghg or 0)

def calculate_monthly_energy_cost(monthly_kwh, monthly_peak_kw, monthly_therms):
    base_charge = 0
    demand_rate = 0
    energy_rate = 0.09
    gas_rate = 1.33
    tax_rate = 0

    elec_cost = (
        base_charge +
        demand_rate * monthly_peak_kw +
        energy_rate * monthly_kwh
    )
    elec_cost_total = elec_cost * (1 + tax_rate)
    gas_cost_total = monthly_therms * gas_rate

    return elec_cost_total + gas_cost_total

def plot_multi_simulation(sql_paths, year, labels_full=None, labels_short=None, output_dir=None, city_name=None):
    if labels_full is None:
        labels_full = [f"Run_{i+1}" for i in range(len(sql_paths))]
    if labels_short is None:
        labels_short = labels_full

    colors = px.colors.qualitative.Plotly
    colors = colors * ((len(sql_paths) // len(colors)) + 1)

    sim_results = [extract_simulation_data(sql, year) for sql in sql_paths]
    
    months = sim_results[0].index
    x_labels = months.strftime('%b')
    costs = []

    for df in sim_results:
        kwh = df["Electricity_MWh"] * 1000
        peak_kw = df["Max_Electricity_MW"] * 1000
        therms = df["Gas_Therms"]
        monthly_cost = calculate_monthly_energy_cost(kwh, peak_kw, therms)
        costs.append(monthly_cost)

    annual_elec = [df["Electricity_MWh"].sum() * 1000 for df in sim_results]
    annual_gas = [df["Gas_Therms"].sum() for df in sim_results]
    annual_cost = [c.sum() for c in costs]

    clean_labels = []
    for label in labels_full:
        if "2 Speed - Lab Data - Electric Backup" in label:
            clean_labels.append("NRELData-Electric")
        elif "2 Speed - Lab Data - Gas Backup" in label:
            clean_labels.append("NRELData-DualFuel")
        elif "HPC curves - Electric Backup" in label:
            clean_labels.append("HPC-Electric")
        elif "HPC curves - Gas Backup" in label or "CCHP curves - Gas Backup" in label or "HP Challenge curves - Gas Backup" in label:
            clean_labels.append("HPC-DualFuel")
        else:
            clean_labels.append(label)

    elec_title = "Energy (kWh)<br>" + " | ".join([f"{l}: {v:,.0f}" for l, v in zip(clean_labels[:3], annual_elec[:3])]) + "<br>" + " | ".join([f"{l}: {v:,.0f}" for l, v in zip(clean_labels[3:], annual_elec[3:])])
    gas_title = "Gas (Therms)<br>" + " | ".join([f"{l}: {v:,.0f}" for l, v in zip(clean_labels[:3], annual_gas[:3])]) + "<br>" + " | ".join([f"{l}: {v:,.0f}" for l, v in zip(clean_labels[3:], annual_gas[3:])])
    peak_title = "Peak Electricity (kW)"
    cost_k = [c / 1000 for c in annual_cost]
    cost_title = "Energy Cost<br>" + " | ".join([f"{l}: &#36;{v:,.1f}k" for l, v in zip(clean_labels[:3], cost_k[:3])]) + "<br>" + " | ".join([f"{l}: &#36;{v:,.1f}k" for l, v in zip(clean_labels[3:], cost_k[3:])])

    fig = make_subplots(
        rows=2, cols=2, shared_xaxes=False,
        subplot_titles=[elec_title, gas_title, peak_title, cost_title],
        vertical_spacing=0.35,
        horizontal_spacing=0.15
    )

    subplot_map = {'Electricity': (1, 1), 'Gas': (1, 2), 'Peak': (2, 1), 'Cost': (2, 2)}

    legend_labels = []
    for label in labels_full:
        if "2 Speed - Lab Data - Electric Backup" in label:
            legend_labels.append("2 Speed NREL Data - Electric")
        elif "2 Speed - Lab Data - Gas Backup" in label:
            legend_labels.append("2 Speed NREL Data - Dual Fuel")
        elif "HPC curves - Electric Backup" in label:
            legend_labels.append("HP Challenge - Electric")
        elif "HPC curves - Gas Backup" in label or "CCHP curves - Gas Backup" in label or "HP Challenge curves - Gas Backup" in label:
            legend_labels.append("HP Challenge - Dual Fuel")
        else:
            legend_labels.append(label)
    
    for df, legend_label, color in zip(sim_results, legend_labels, colors):
        fig.add_trace(go.Bar(x=x_labels, y=df["Electricity_MWh"] * 1000, name=legend_label, marker_color=color),
                      row=subplot_map['Electricity'][0], col=subplot_map['Electricity'][1])
        fig.add_trace(go.Bar(x=x_labels, y=df["Gas_Therms"], name=legend_label, marker_color=color, showlegend=False),
                      row=subplot_map['Gas'][0], col=subplot_map['Gas'][1])
        fig.add_trace(go.Bar(x=x_labels, y=df["Max_Electricity_MW"] * 1000, name=legend_label, marker_color=color, showlegend=False),
                      row=subplot_map['Peak'][0], col=subplot_map['Peak'][1])

    for cost, legend_label, color in zip(costs, legend_labels, colors):
        fig.add_trace(go.Bar(x=x_labels, y=cost, name=legend_label, marker_color=color, showlegend=False),
                      row=subplot_map['Cost'][0], col=subplot_map['Cost'][1])

    main_title = f"Simulation Comparison - {city_name}" if city_name else "Simulation Comparison"

    fig.update_layout(
        height=800, width=960, title=dict(text=main_title, y=0.98, x=0.5, xanchor='center', yanchor='top', font=dict(size=16)),
        template='simple_white', barmode='group', showlegend=True,
        legend=dict(orientation="h", yanchor="top", y=0.50, xanchor="center", x=0.5, font=dict(size=11), bgcolor="rgba(255,255,255,0.9)", bordercolor="gray", borderwidth=1),
        margin=dict(t=80)
    )

    for i in range(4):
        fig.layout.annotations[i].font.size = 10

    timestamp = pd.Timestamp.now().strftime("%Y%m%d%H%M")
    safe_city = city_name.replace(' ', '_') if city_name else 'city'
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
        jpeg_path = os.path.join(output_dir, f"{safe_city}_comparison_scenarios.jpeg")
        try:
            fig.write_image(jpeg_path, format="jpeg", scale=2, width=960, height=720)
        except Exception: pass
    fig.show()

def get_city_weather_files(city_name):
    if city_name not in city_region_state:
        return None
    folder_name = 'USA_PA_Lancaster.AP.725116_TMYx' if city_name == 'Lancaster_TMYx' else city_name
    city_folder_path = os.path.join(base_weather_path, folder_name)
    if not os.path.exists(city_folder_path):
        return None
    epw_file = ddy_file = None
    for filename in os.listdir(city_folder_path):
        if filename.lower().endswith('.epw'):
            epw_file = os.path.join(city_folder_path, filename)
        elif filename.lower().endswith('.ddy'):
            ddy_file = os.path.join(city_folder_path, filename)
    return {'epw': epw_file, 'ddy': ddy_file}

def create_city_scenarios(city, base_run_dir, measure_dir_path, seed_model_path, overwrite_existing=False):
    wf = get_city_weather_files(city)
    if wf is None or wf['epw'] is None:
        print(f"❌ Weather files not found for {city}")
        return
    epw_path, ddy_path = wf['epw'], wf['ddy']

    info = city_region_state[city]
    grid_state = info['state']
    grid_region = info['cambium_region']

    scenarios = [
        {"name": "Baseline", "upgrade_args": {"hprtu_scenario": "Baseline", "backup_ht_fuel_scheme": "electric_resistance_backup", "performance_oversizing_factor": 0, "htg_sizing_option": "0F", "clg_oversizing_estimate": 1, "htg_to_clg_hp_ratio": 1.0, "hp_min_comp_lockout_temp_f": 0.0, "hr": False, "dcv": False, "econ": False, "sizing_run": False, "debug_verbose": False}},
        {"name": "2 Speed - Lab Data - Electric Backup", "upgrade_args": {"hprtu_scenario": "two_speed_lab_data", "backup_ht_fuel_scheme": "electric_resistance_backup", "performance_oversizing_factor": 0, "htg_sizing_option": "0F", "clg_oversizing_estimate": 1, "htg_to_clg_hp_ratio": 1.0, "hp_min_comp_lockout_temp_f": 0.0, "hr": False, "dcv": False, "econ": False, "sizing_run": False, "debug_verbose": False}},
        {"name": "2 Speed - Lab Data - Gas Backup", "upgrade_args": {"hprtu_scenario": "two_speed_lab_data", "backup_ht_fuel_scheme": "match_original_primary_heating_fuel", "performance_oversizing_factor": 0, "htg_sizing_option": "0F", "clg_oversizing_estimate": 1, "htg_to_clg_hp_ratio": 1.0, "hp_min_comp_lockout_temp_f": 30.0, "hr": False, "dcv": False, "econ": False, "sizing_run": False, "debug_verbose": False}},
        {"name": "HPC curves - Electric Backup", "upgrade_args": {"hprtu_scenario": "cchpc_2027_spec", "backup_ht_fuel_scheme": "electric_resistance_backup", "performance_oversizing_factor": 0, "htg_sizing_option": "0F", "clg_oversizing_estimate": 1, "htg_to_clg_hp_ratio": 1.0, "hp_min_comp_lockout_temp_f": -10.0, "hr": False, "dcv": False, "econ": False, "sizing_run": False, "debug_verbose": False}},
        {"name": "HP Challenge curves - Gas Backup", "upgrade_args": {"hprtu_scenario": "cchpc_2027_spec", "backup_ht_fuel_scheme": "match_original_primary_heating_fuel", "performance_oversizing_factor": 0, "htg_sizing_option": "0F", "clg_oversizing_estimate": 1, "htg_to_clg_hp_ratio": 1.0, "hp_min_comp_lockout_temp_f": -10.0, "hr": False, "dcv": False, "econ": False, "sizing_run": False, "debug_verbose": False}}
    ]

    city_run_dir = os.path.join(base_run_dir, city)
    os.makedirs(city_run_dir, exist_ok=True)

    for scenario in scenarios:
        run_dir = os.path.join(city_run_dir, scenario["name"])
        sql_output_path = os.path.join(run_dir, "run", "eplusout.sql")

        if os.path.exists(sql_output_path) and not overwrite_existing:
            print(f"⏭️ Skipping {scenario['name']} - simulation already exists.")
            continue

        os.makedirs(run_dir, exist_ok=True)
        osw_path = os.path.join(run_dir, "in.osw")
        steps = [
            {"measure_dir_name": "target_upgrade_measure", "arguments": scenario["upgrade_args"]},
            {"measure_dir_name": EMISSIONS_MEASURE_DIR_NAME, "arguments": {"grid_region": grid_region, "grid_state": grid_state, "emissions_scenario": EMISSIONS_SCENARIO}}
        ]

        osw = {"seed_file": seed_model_path, "weather_file": epw_path, "measure_paths": [measure_dir_path], "steps": steps, "name": scenario["name"]}
        with open(osw_path, "w") as f: json.dump(osw, f, indent=2)

        try:
            subprocess.run(["openstudio", "run", "-w", osw_path], check=True, capture_output=True, text=True)
            print(f"✅ {scenario['name']} completed.")
        except Exception as e:
            print(f"❌ {scenario['name']} failed.")

# === DATA DICTIONARIES ===
city_region_state = {
    'Miami': {'state': 'FL', 'climate_zone': '1A', 'cambium_region': 'FRCCc'},
    'Houston': {'state': 'TX', 'climate_zone': '2A', 'cambium_region': 'ERCTc'},
    'Phoenix': {'state': 'AZ', 'climate_zone': '2B', 'cambium_region': 'AZNMc'},
    'Atlanta': {'state': 'GA', 'climate_zone': '3A', 'cambium_region': 'SRSOc'},
    'ElPaso': {'state': 'TX', 'climate_zone': '3B', 'cambium_region': 'AZNMc'},
    'SanFrancisco': {'state': 'CA', 'climate_zone': '3C', 'cambium_region': 'CAMXc'},
    'Lancaster': {'state': 'PA', 'climate_zone': '4A', 'cambium_region': 'RFCEc'},
    'Amarillo': {'state': 'TX', 'climate_zone': '4B', 'cambium_region': 'SPSOc'},
    'Portland': {'state': 'OR', 'climate_zone': '4C', 'cambium_region': 'NWPPc'},
    'Chicago': {'state': 'IL', 'climate_zone': '5A', 'cambium_region': 'RFCWc'},
    'Denver': {'state': 'CO', 'climate_zone': '5B', 'cambium_region': 'RMPAc'},
    'PortAngeles': {'state': 'WA', 'climate_zone': '5C', 'cambium_region': 'NWPPc'},
    'Minneapolis': {'state': 'MN', 'climate_zone': '6A', 'cambium_region': 'MROEc'},
    'Helena': {'state': 'MT', 'climate_zone': '6B', 'cambium_region': 'NWPPc'},
    'Duluth': {'state': 'MN', 'climate_zone': '7', 'cambium_region': 'MROEc'},
    'Fairbanks': {'state': 'AK', 'climate_zone': '8', 'cambium_region': 'AKGD'},
}

# === FIXED PATHS ===
seed_model_path = '/Users/cbianchi/Documents/GitHub/Target_HPs/Modeling_resources/Target_BASELINE_seed.osm'
base_weather_path = '/Users/cbianchi/Documents/GitHub/Target_HPs/Modeling_resources/weather'
measure_dir_path = '/Users/cbianchi/Documents/GitHub/Target_HPs/Modeling_resources/measures'
base_run_dir = '/Users/cbianchi/Documents/GitHub/Target_HPs/Enterprise/simulation_run'
EMISSIONS_SCENARIO = 'LRMER_MidCase_30'
EMISSIONS_MEASURE_DIR_NAME = 'emissions_reporting'

# === EXECUTION LOGIC ===
PLOT_ONLY = False # Set to False to run the PortAngeles scenarios
OVERWRITE_EXISTING = False
TARGET_CITY = 'PortAngeles'

if TARGET_CITY in city_region_state:
    # Filter to only the target city
    cities_to_process = {TARGET_CITY: city_region_state[TARGET_CITY]}
    
    for city, info in cities_to_process.items():
        if not PLOT_ONLY:
            print(f"🚀 Running scenarios for {city}...")
            create_city_scenarios(city, base_run_dir, measure_dir_path, seed_model_path, overwrite_existing=OVERWRITE_EXISTING)
        
        city_run_dir = os.path.join(base_run_dir, city)
        plots_dir = os.path.join(os.path.dirname(base_run_dir), 'plots')
        os.makedirs(plots_dir, exist_ok=True)

        sql_paths, labels_full, labels_short = [], [], []
        for name in ["Baseline", "2 Speed - Lab Data - Electric Backup", "2 Speed - Lab Data - Gas Backup", "HPC curves - Electric Backup", "HP Challenge curves - Gas Backup"]:
            sql_path = os.path.join(city_run_dir, name, "run", "eplusout.sql")
            if os.path.exists(sql_path):
                sql_paths.append(sql_path)
                labels_full.append(name)
                labels_short.append(name.split(' ')[0])

        if sql_paths:
            city_with_zone = f"{city} (Climate Zone {info['climate_zone']})"
            plot_multi_simulation(sql_paths, 2020, labels_full, labels_short, output_dir=plots_dir, city_name=city_with_zone)
            print(f"📈 Resulting plots saved in {plots_dir}")
else:
    print(f"Error: {TARGET_CITY} not found in dictionary.")


🚀 Running scenarios for PortAngeles...
⏭️ Skipping Baseline - simulation already exists.
⏭️ Skipping 2 Speed - Lab Data - Electric Backup - simulation already exists.
✅ 2 Speed - Lab Data - Gas Backup completed.
⏭️ Skipping HPC curves - Electric Backup - simulation already exists.
✅ HP Challenge curves - Gas Backup completed.


📈 Resulting plots saved in /Users/cbianchi/Documents/GitHub/Target_HPs/Enterprise/plots


### Configuration & Control Flags


In [3]:
# ============================================================================
# ANALYSIS CONTROL FLAGS
# ============================================================================
# Set these flags to control whether to recalculate or use existing data

# Cost calculation settings
RECALCULATE_COSTS = True  # If True: recalculate all energy costs from SQL files
                           # If False: load existing cost data from checkpoint/CSV

# File paths for loading existing data
EXISTING_COSTS_FILE = '/Users/cbianchi/Documents/GitHub/Target_HPs/Enterprise/store_costs_checkpoint.csv'
# Alternatively, use the full analysis file:
# EXISTING_COSTS_FILE = '/Users/cbianchi/Documents/GitHub/Target_HPs/Enterprise/Target_Stores_Energy_Cost_Analysis.csv'

# Normalization settings
RECALCULATE_NORMALIZED_COSTS = True  # If True: recalculate normalized costs and savings %
                                       # If False: use existing normalized columns if present

# County FIPS settings  
RECALCULATE_COUNTY_FIPS = False  # If True: re-copy county data from df_stores
                                  # If False: use existing county columns if present

# Verification file settings
CREATE_VERIFICATION_FILE = True  # If True: create Excel file for manual verification
                                  # If False: skip verification file creation

# Display settings
VERBOSE_OUTPUT = False  # If True: show detailed progress messages
                       # If False: show only essential messages

print("📋 Analysis Control Flags:")
print(f"   RECALCULATE_COSTS: {RECALCULATE_COSTS}")
print(f"   RECALCULATE_NORMALIZED_COSTS: {RECALCULATE_NORMALIZED_COSTS}")
print(f"   RECALCULATE_COUNTY_FIPS: {RECALCULATE_COUNTY_FIPS}")
print(f"   CREATE_VERIFICATION_FILE: {CREATE_VERIFICATION_FILE}")
print(f"   VERBOSE_OUTPUT: {VERBOSE_OUTPUT}")

📋 Analysis Control Flags:
   RECALCULATE_COSTS: True
   RECALCULATE_NORMALIZED_COSTS: True
   RECALCULATE_COUNTY_FIPS: False
   CREATE_VERIFICATION_FILE: True
   VERBOSE_OUTPUT: False


### Analysis Workflow


### Data Loading


In [4]:
# Helper functions for energy cost analysis

def get_hourly_loads_from_sql(sql_path, year=2020):
    """
    Extract hourly electricity (kW) and gas (therms/hr) loads from SQL file.
    Returns DataFrame with DateTimeIndex and columns: Electricity_kW, Gas_Therms_per_Hr
    """
    # Get hourly energy in MWh and Therms
    elec_mwh = get_energy_from_sql(sql_path, year, 'Electricity')
    gas_therms = get_energy_from_sql(sql_path, year, 'NaturalGas')
    
    # Convert MWh to kW (average power over the hour)
    elec_kw = elec_mwh * 1000
    
    df = pd.DataFrame({
        'Electricity_kW': elec_kw,
        'Gas_Therms_per_Hr': gas_therms
    })
    
    return df

def calculate_monthly_peak_demand_5am_10pm(hourly_loads):
    """
    Calculate the maximum hourly load that occurs between 5am-10pm for each month.
    
    Args:
        hourly_loads: DataFrame with DateTimeIndex and 'Electricity_kW' column
    
    Returns:
        Series with monthly peak demand (kW) during 5am-10pm period
    """
    # Filter to 5am-10pm hours (5-21 inclusive, as hour 21 = 9pm-10pm)
    mask = (hourly_loads.index.hour >= 5) & (hourly_loads.index.hour < 22)
    filtered = hourly_loads[mask]
    
    # Group by month and find max
    monthly_peak = filtered.groupby(filtered.index.to_period('M'))['Electricity_kW'].max()
    
    return monthly_peak

def highlight_counties(df):
    """
    Apply alternating background colors to rows based on county grouping.
    Stores in the same county will have the same background color.
    """
    # Sort by county for better visualization
    df_sorted = df.sort_values(['County_FIPS', 'County_Name'], na_position='last').copy()
    
    # Create color mapping for counties
    unique_counties = df_sorted['County_FIPS'].dropna().unique()
    
    # Define two alternating colors (light blue and light yellow)
    colors = ['background-color: #E3F2FD', 'background-color: #FFF9C4']
    
    # Create a mapping from county FIPS to color
    county_colors = {county: colors[i % 2] for i, county in enumerate(unique_counties)}
    
    # Apply styling
    def style_row(row):
        if pd.isna(row['County_FIPS']):
            return ['background-color: #FFCDD2'] * len(row)  # Light red for missing FIPS
        else:
            return [county_colors[row['County_FIPS']]] * len(row)
    
    styled = df_sorted.style.apply(style_row, axis=1)
    
    return styled, df_sorted

# Main cost calculation function with optimization and checkpointing

def calculate_store_energy_costs(df_stores, df_bills, city_region_state, base_run_dir, year=2020, 
                                 checkpoint_file='store_costs_checkpoint.csv', save_interval=50):
    """
    Calculate energy costs for all stores across all scenarios with checkpointing.
    
    OPTIMIZATIONS:
    - Pre-loads all SQL data once per city (not per store)
    - Saves progress every save_interval stores
    - Resumes from last checkpoint if interrupted
    
    Args:
        checkpoint_file: Path to save intermediate results
        save_interval: Save progress every N stores
    
    Returns:
        DataFrame with added columns for each scenario's costs and energy metrics
    """
    # Scenario names matching your simulation structure
    scenarios = [
        "Baseline",
        "2 Speed - Lab Data - Electric Backup",
        "2 Speed - Lab Data - Gas Backup",
        "HPC curves - Electric Backup",
        "HP Challenge curves - Gas Backup"
    ]
    
    # Short names for column labels
    scenario_short_names = {
        "Baseline": "Baseline",
        "2 Speed - Lab Data - Electric Backup": "NREL_Electric",
        "2 Speed - Lab Data - Gas Backup": "NREL_DualFuel",
        "HPC curves - Electric Backup": "HPC_Electric",
        "HP Challenge curves - Gas Backup": "HPC_DualFuel"
    }
    
    # Create climate zone to city mapping
    cz_to_city = {info['climate_zone']: city for city, info in city_region_state.items()}
    
    # Pre-cache utility rates by state for faster lookup
    rates_cache = {}
    for state in df_bills['LOCATION STATE/PROVINCE'].unique():
        state_rates = df_bills[df_bills['LOCATION STATE/PROVINCE'] == state]
        if not state_rates.empty:
            # Use fillna(0) for gas rate since some states (HI, FL) have no gas infrastructure
            gas_rate = state_rates.iloc[0]['Gas Rate ($/Therm)']
            rates_cache[state] = {
                'elec_consumption': state_rates.iloc[0]['Electric Consumption Rate ($/kWh)'],
                'elec_demand': state_rates.iloc[0]['Electric Demand Rate ($/kW)'],
                'gas': gas_rate if pd.notna(gas_rate) else 0.0
            }
    
    # PRE-LOAD ALL SQL DATA (major speedup!)
    # Check if cached CSV exists first
    cache_csv_path = os.path.join(os.path.dirname(base_run_dir), 'city_scenario_cache.csv')
    
    if os.path.exists(cache_csv_path):
        print(f"📂 Loading pre-computed data from {cache_csv_path}")
        cache_df = pd.read_csv(cache_csv_path)
        
        # Rebuild nested dictionary from CSV
        city_scenario_cache = {}
        for _, row in cache_df.iterrows():
            city = row['city']
            scenario = row['scenario']
            if city not in city_scenario_cache:
                city_scenario_cache[city] = {}
            city_scenario_cache[city][scenario] = {
                'annual_elec_kwh': row['annual_elec_kwh'],
                'annual_gas_therms': row['annual_gas_therms'],
                'monthly_peak_sum': row['monthly_peak_sum'],
                'peak_demand_kw': row['peak_demand_kw']
            }
        print(f"✅ Loaded cached data for {len(city_scenario_cache)} cities from CSV")
    else:
        print("🔄 Pre-loading all simulation data from SQL files...")
        city_scenario_cache = {}
        cache_data = []  # For saving to CSV
        
        for city in cz_to_city.values():
            city_scenario_cache[city] = {}
            for scenario_name in scenarios:
                sql_path = os.path.join(base_run_dir, city, scenario_name, "run", "eplusout.sql")
                if os.path.exists(sql_path):
                    try:
                        hourly_loads = get_hourly_loads_from_sql(sql_path, year)
                        monthly_peak_kw = calculate_monthly_peak_demand_5am_10pm(hourly_loads)
                        
                        city_scenario_cache[city][scenario_name] = {
                            'annual_elec_kwh': hourly_loads['Electricity_kW'].sum(),
                            'annual_gas_therms': hourly_loads['Gas_Therms_per_Hr'].sum(),
                            'monthly_peak_sum': monthly_peak_kw.sum(),
                            'peak_demand_kw': monthly_peak_kw.max()
                        }
                        
                        # Add to cache data for CSV
                        cache_data.append({
                            'city': city,
                            'scenario': scenario_name,
                            'annual_elec_kwh': city_scenario_cache[city][scenario_name]['annual_elec_kwh'],
                            'annual_gas_therms': city_scenario_cache[city][scenario_name]['annual_gas_therms'],
                            'monthly_peak_sum': city_scenario_cache[city][scenario_name]['monthly_peak_sum'],
                            'peak_demand_kw': city_scenario_cache[city][scenario_name]['peak_demand_kw']
                        })
                    except Exception as e:
                        print(f"⚠️  Failed to load {city}/{scenario_name}: {e}")
        
        # Save to CSV for next time
        if cache_data:
            cache_df = pd.DataFrame(cache_data)
            cache_df.to_csv(cache_csv_path, index=False)
            print(f"💾 Saved cache to {cache_csv_path}")
        
        print(f"✅ Pre-loaded data for {len(city_scenario_cache)} cities from SQL")
    
    # Check for existing checkpoint
    checkpoint_path = os.path.join(os.path.dirname(base_run_dir), checkpoint_file)
    if os.path.exists(checkpoint_path):
        print(f"📂 Loading checkpoint from {checkpoint_path}")
        df_result = pd.read_csv(checkpoint_path)
        # Find first row without cost data
        start_idx = 0
        for idx in df_result.index:
            if pd.isna(df_result.loc[idx, 'Baseline_Total_Energy_Cost']):
                start_idx = idx
                break
        else:
            start_idx = len(df_result)
        print(f"▶️  Resuming from store index {start_idx}")
    else:
        df_result = df_stores.copy()
        start_idx = 0
        print("🆕 Starting fresh analysis")
    
    # Process each store
    stores_processed = 0
    for idx in range(start_idx, len(df_result)):
        store_row = df_result.iloc[idx]
        store_cz = str(store_row['climate zone']).strip()
        store_state = store_row['State']
        
        # Find matching city from climate zone
        if store_cz not in cz_to_city:
            print(f"⚠️  Store {idx}: Climate zone '{store_cz}' not found")
            continue
        
        city = cz_to_city[store_cz]
        
        # Get cached rates
        if store_state not in rates_cache:
            print(f"⚠️  Store {idx}: No utility rates for state '{store_state}'")
            continue
        
        rates = rates_cache[store_state]
        
        # Check if city data is cached
        if city not in city_scenario_cache:
            print(f"⚠️  Store {idx}: No simulation data for city '{city}'")
            continue
        
        # Process each scenario using cached data
        for scenario_name in scenarios:
            short_name = scenario_short_names[scenario_name]
            
            if scenario_name not in city_scenario_cache[city]:
                continue
            
            data = city_scenario_cache[city][scenario_name]
            
            # Calculate costs using cached data
            annual_elec_consumption_cost = data['annual_elec_kwh'] * rates['elec_consumption']
            annual_elec_demand_cost = data['monthly_peak_sum'] * rates['elec_demand']
            annual_gas_cost = data['annual_gas_therms'] * rates['gas']
            annual_total_cost = annual_elec_consumption_cost + annual_elec_demand_cost + annual_gas_cost
            
            # Add columns to result DataFrame
            df_result.loc[idx, f'{short_name}_Electric_Consumption_Cost'] = annual_elec_consumption_cost
            df_result.loc[idx, f'{short_name}_Electric_Demand_Cost'] = annual_elec_demand_cost
            df_result.loc[idx, f'{short_name}_Gas_Cost'] = annual_gas_cost
            df_result.loc[idx, f'{short_name}_Total_Energy_Cost'] = annual_total_cost
            df_result.loc[idx, f'{short_name}_Electricity_kWh'] = data['annual_elec_kwh']
            df_result.loc[idx, f'{short_name}_Gas_Therms'] = data['annual_gas_therms']
            df_result.loc[idx, f'{short_name}_Peak_Demand_kW'] = data['peak_demand_kw']
        
        stores_processed += 1
        
        # Periodic status update
        if stores_processed % 100 == 0:
            print(f"✅ Processed {stores_processed} stores (current: Store {idx}, {store_state}, CZ {store_cz})")
        
        # Save checkpoint
        if stores_processed % save_interval == 0:
            df_result.to_csv(checkpoint_path, index=False)
            print(f"💾 Checkpoint saved at store {idx} ({stores_processed} processed)")
    
    return df_result



# Load store data and utility rates
excel_path_stores = '/Users/cbianchi/Documents/GitHub/Target_HPs/Enterprise/resources/Target All Location Store Attributes_with_cambium.xlsx'
csv_path_rates = '/Users/cbianchi/Documents/GitHub/Target_HPs/Enterprise/resources/combined_bills_by_state.csv'

df_bills = pd.read_csv(csv_path_rates)
df_stores = pd.read_excel(excel_path_stores, sheet_name=0)

print(f"✅ Loaded {len(df_stores)} stores from Excel")
print(f"✅ Loaded utility rates for {len(df_bills)} states")

#

✅ Loaded 1987 stores from Excel
✅ Loaded utility rates for 51 states


In [5]:
# ============================================================================
# STEP 1: Calculate or Load Energy Costs
# ============================================================================

# Create climate zone to city mapping
cz_to_city = {info['climate_zone']: city for city, info in city_region_state.items()}

if RECALCULATE_COSTS:
    print("🔄 RECALCULATING energy costs from SQL files...")
    df_stores_with_costs = calculate_store_energy_costs(
        df_stores=df_stores,
        df_bills=df_bills,
        city_region_state=city_region_state,
        base_run_dir=base_run_dir,
        year=2020,
        checkpoint_file='store_costs_checkpoint.csv',
        save_interval=50
    )
    print("\n" + "="*80)
    print("✅ Analysis complete!")
    print(f"   Total stores with cost data: {df_stores_with_costs['Baseline_Total_Energy_Cost'].notna().sum()}")
    print("="*80)
else:
    print("📂 LOADING existing cost data...")
    if os.path.exists(EXISTING_COSTS_FILE):
        df_stores_with_costs = pd.read_csv(EXISTING_COSTS_FILE)
        print(f"✅ Loaded data from: {EXISTING_COSTS_FILE}")
        print(f"   Total stores: {len(df_stores_with_costs)}")
        print(f"   Stores with cost data: {df_stores_with_costs['Baseline_Total_Energy_Cost'].notna().sum()}")
        print(f"   Total columns: {len(df_stores_with_costs.columns)}")
    else:
        print(f"❌ File not found: {EXISTING_COSTS_FILE}")
        print(f"   Set RECALCULATE_COSTS=True to generate new data")
        print(f"   Or update EXISTING_COSTS_FILE path to point to an existing CSV")
        df_stores_with_costs = df_stores.copy()

# ============================================================================
# STEP 2: Add County FIPS Data
# ============================================================================

if RECALCULATE_COUNTY_FIPS or 'County_FIPS' not in df_stores_with_costs.columns:
    if VERBOSE_OUTPUT:
        print("\n📋 Checking for County FIPS data...")
    
    county_cols_in_stores = [col for col in df_stores.columns if 'county' in col.lower() or 'fips' in col.lower()]
    
    if len(county_cols_in_stores) > 0:
        print("🔄 Copying County FIPS data from original store data...")
        for col in county_cols_in_stores:
            df_stores_with_costs[col] = df_stores[col]
        print(f"✅ Copied {len(county_cols_in_stores)} county-related columns")
    else:
        print("⚠️  No county-related columns found in df_stores")
else:
    if VERBOSE_OUTPUT:
        print("\n✅ Using existing County FIPS data")

# ============================================================================
# STEP 3: Calculate Normalized Costs and Savings
# ============================================================================

# Model area constants
MODEL_AREA_M2 = 11567.67
MODEL_AREA_SQFT = MODEL_AREA_M2 * 10.7639
scenario_short_names = ["Baseline", "NREL_Electric", "NREL_DualFuel", "HPC_Electric", "HPC_DualFuel"]

# Check if normalized columns already exist
normalized_cols_exist = all(
    f'{scenario}_Normalized_Total_Cost' in df_stores_with_costs.columns 
    for scenario in scenario_short_names
)
savings_cols_exist = all(
    f'{scenario}_Savings_vs_Baseline_Percent' in df_stores_with_costs.columns 
    for scenario in scenario_short_names[1:]
)

if RECALCULATE_NORMALIZED_COSTS or not (normalized_cols_exist and savings_cols_exist):
    print("\n🔄 Calculating normalized costs and savings percentages...")
    
    # Normalized costs
    for scenario in scenario_short_names:
        cost_col = f'{scenario}_Total_Energy_Cost'
        normalized_col = f'{scenario}_Normalized_Total_Cost'
        
        if cost_col in df_stores_with_costs.columns:
            df_stores_with_costs[normalized_col] = (df_stores_with_costs[cost_col] * df_stores_with_costs['Building Area (Sq Ft)'] / MODEL_AREA_SQFT)
    
    # Savings percentages
    baseline_col = 'Baseline_Normalized_Total_Cost'
    for scenario in scenario_short_names[1:]:
        cost_col = f'{scenario}_Normalized_Total_Cost'
        savings_col = f'{scenario}_Savings_vs_Baseline_Percent'
        
        if cost_col in df_stores_with_costs.columns:
            df_stores_with_costs[savings_col] = (
                (df_stores_with_costs[baseline_col] - df_stores_with_costs[cost_col]) / 
                df_stores_with_costs[baseline_col] * 100
            )
    
    print("✅ Normalized costs and savings calculated")
else:
    if VERBOSE_OUTPUT:
        print("\n✅ Using existing normalized costs and savings")

# ============================================================================
# STEP 4: Display Summary Statistics
# ============================================================================

if VERBOSE_OUTPUT:
    print("\n" + "="*80)
    print("NORMALIZED TOTAL COSTS ($/sqft/year):")
    print("="*80)
    for scenario in scenario_short_names:
        norm_col = f'{scenario}_Normalized_Total_Cost_per_sqft'
        if norm_col in df_stores_with_costs.columns:
            mean_val = df_stores_with_costs[norm_col].mean()
            median_val = df_stores_with_costs[norm_col].median()
            print(f"{scenario:20s}: Mean=${mean_val:.3f}/sqft/yr, Median=${median_val:.3f}/sqft/yr")
    
    print("\n" + "="*80)
    print("SAVINGS vs BASELINE (%):")
    print("="*80)
    for scenario in scenario_short_names[1:]:
        savings_col = f'{scenario}_Savings_vs_Baseline_Percent'
        if savings_col in df_stores_with_costs.columns:
            mean_val = df_stores_with_costs[savings_col].mean()
            median_val = df_stores_with_costs[savings_col].median()
            print(f"{scenario:20s}: Mean={mean_val:+.1f}%, Median={median_val:+.1f}%")
    
    print("\n" + "="*80)
    
    # Show sample store
    for idx, row in df_stores_with_costs.iterrows():
        if pd.notna(row.get('Baseline_Total_Energy_Cost')):
            print(f"\nSample Store {idx} - {row['State']}, {row['Building Area (Sq Ft)']:.0f} sqft:")
            print(f"  Baseline: ${row['Baseline_Total_Energy_Cost']:,.0f}/yr (${row['Baseline_Normalized_Total_Cost_per_sqft']:.3f}/sqft/yr)")
            
            for scenario in scenario_short_names[1:]:
                cost = row.get(f'{scenario}_Total_Energy_Cost', 0)
                norm = row.get(f'{scenario}_Normalized_Total_Cost_per_sqft', 0)
                savings = row.get(f'{scenario}_Savings_vs_Baseline_Percent', 0)
                print(f"  {scenario:15s}: ${cost:,.0f}/yr (${norm:.3f}/sqft/yr) - {savings:+.1f}% vs Baseline")
            break

print("\n" + "="*80)
print("📊 ANALYSIS COMPLETE")
print("="*80)
print(f"Total stores: {len(df_stores_with_costs)}")
print(f"Stores with cost data: {df_stores_with_costs.get('Baseline_Total_Energy_Cost', pd.Series()).notna().sum()}")
print(f"Total columns: {len(df_stores_with_costs.columns)}")


# Save complete analysis to CSV
output_csv_path = '/Users/cbianchi/Documents/GitHub/Target_HPs/Enterprise/Target_Stores_Energy_Cost_Analysis.csv'
df_stores_with_costs.to_csv(output_csv_path, index=False)

df_stores_with_costs.head()

🔄 RECALCULATING energy costs from SQL files...
🔄 Pre-loading all simulation data from SQL files...
💾 Saved cache to /Users/cbianchi/Documents/GitHub/Target_HPs/Enterprise/city_scenario_cache.csv
✅ Pre-loaded data for 16 cities from SQL
🆕 Starting fresh analysis
💾 Checkpoint saved at store 49 (50 processed)
✅ Processed 100 stores (current: Store 99, MI, CZ 5A)
💾 Checkpoint saved at store 99 (100 processed)
💾 Checkpoint saved at store 149 (150 processed)
✅ Processed 200 stores (current: Store 199, FL, CZ 2A)
💾 Checkpoint saved at store 199 (200 processed)
💾 Checkpoint saved at store 249 (250 processed)
✅ Processed 300 stores (current: Store 299, IA, CZ 6A)
💾 Checkpoint saved at store 299 (300 processed)
💾 Checkpoint saved at store 349 (350 processed)
✅ Processed 400 stores (current: Store 399, GA, CZ 3A)
💾 Checkpoint saved at store 399 (400 processed)
💾 Checkpoint saved at store 449 (450 processed)
✅ Processed 500 stores (current: Store 499, MD, CZ 4A)
💾 Checkpoint saved at store 499 (50

/var/folders/cj/vhw1_9g1151dh76sjlgdyq19k7f3h5/T/ipykernel_37813/291813859.py:149: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  print(f"Stores with cost data: {df_stores_with_costs.get('Baseline_Total_Energy_Cost', pd.Series()).notna().sum()}")


,Location,climate zone,Location Name,Status,Format,Prototype,Site Type Description,Levels,Total Stock Area (Sq Ft),Total Occupied Area (Sq Ft),...,HPC_DualFuel_Peak_Demand_kW,Baseline_Normalized_Total_Cost,NREL_Electric_Normalized_Total_Cost,NREL_DualFuel_Normalized_Total_Cost,HPC_Electric_Normalized_Total_Cost,HPC_DualFuel_Normalized_Total_Cost,NREL_Electric_Savings_vs_Baseline_Percent,NREL_DualFuel_Savings_vs_Baseline_Percent,HPC_Electric_Savings_vs_Baseline_Percent,HPC_DualFuel_Savings_vs_Baseline_Percent
0,T0003,6A,Crystal,Open,Pfresh,P19,Strip Mall,1.0,30268,143771,...,351.428185,249468.345424,295980.198153,250589.468678,268140.117099,244253.929274,-18.644391,-0.449405,-7.484626,2.090212
1,T0004,7,Duluth,Open,Pfresh,P22,Freestanding,1.0,32438,155559,...,333.433823,276743.979081,349543.199924,278620.113564,323064.195224,275432.291792,-26.305620,-0.677931,-16.737570,0.473971
2,T0005,6A,Bloomington 79th and Penn,Open,Pfresh,P19,Strip Mall,1.0,35336,148756,...,351.428185,259172.776612,307493.961396,260337.512066,278570.888628,253755.517321,-18.644391,-0.449405,-7.484626,2.090212
3,T0012,4A,Bridgeton,Open,Pfresh,P22,Freestanding,1.0,35635,136216,...,353.149948,210444.421596,219191.877793,206842.544077,205383.932493,199068.887797,-4.156659,1.711558,2.404668,5.405481
4,T0013,3A,North Dallas,Open,Pfresh,P09MR,Strip Mall,1.0,19989,121991,...,372.747713,183532.118802,182442.814980,177311.012660,171476.669549,169139.694100,0.593522,3.389655,6.568577,7.841911


In [6]:
# =============================================================================
# DIAGNOSTIC: Check Alaska and Hawaii Store Data
# =============================================================================

print("\n" + "="*80)
print("🔍 DIAGNOSTIC: Checking Alaska (AK) and Hawaii (HI) Store Data")
print("="*80)

# Check for AK stores
ak_stores = df_stores_with_costs[df_stores_with_costs['State'] == 'AK']
print(f"\n📍 Alaska (AK) Stores:")
print(f"   Total stores: {len(ak_stores)}")
if len(ak_stores) > 0:
    print(f"   Climate zones: {ak_stores['climate zone'].value_counts().to_dict()}")
    print(f"   Stores with cost data: {ak_stores['Baseline_Total_Energy_Cost'].notna().sum()}")
    print(f"   Stores missing cost data: {ak_stores['Baseline_Total_Energy_Cost'].isna().sum()}")
    # Show a sample
    if ak_stores['Baseline_Total_Energy_Cost'].notna().any():
        sample = ak_stores[ak_stores['Baseline_Total_Energy_Cost'].notna()].iloc[0]
        print(f"   Sample store with data: {sample['Location Name']} (CZ: {sample['climate zone']})")
else:
    print("   ⚠️  NO ALASKA STORES IN DATASET")

# Check for HI stores
hi_stores = df_stores_with_costs[df_stores_with_costs['State'] == 'HI']
print(f"\n🏝️  Hawaii (HI) Stores:")
print(f"   Total stores: {len(hi_stores)}")
if len(hi_stores) > 0:
    print(f"   Climate zones: {hi_stores['climate zone'].value_counts().to_dict()}")
    print(f"   Stores with cost data: {hi_stores['Baseline_Total_Energy_Cost'].notna().sum()}")
    print(f"   Stores missing cost data: {hi_stores['Baseline_Total_Energy_Cost'].isna().sum()}")
    # Show a sample
    if hi_stores['Baseline_Total_Energy_Cost'].notna().any():
        sample = hi_stores[hi_stores['Baseline_Total_Energy_Cost'].notna()].iloc[0]
        print(f"   Sample store with data: {sample['Location Name']} (CZ: {sample['climate zone']})")
else:
    print("   ⚠️  NO HAWAII STORES IN DATASET")

# Check all states with missing cost data
print(f"\n📊 States with missing cost data:")
missing_by_state = df_stores_with_costs[df_stores_with_costs['Baseline_Total_Energy_Cost'].isna()].groupby('State').size().sort_values(ascending=False)
if len(missing_by_state) > 0:
    for state, count in missing_by_state.head(10).items():
        print(f"   {state}: {count} stores")
else:
    print("   ✅ All states have complete cost data!")

print("\n" + "="*80)


🔍 DIAGNOSTIC: Checking Alaska (AK) and Hawaii (HI) Store Data

📍 Alaska (AK) Stores:
   Total stores: 3
   Climate zones: {7: 3}
   Stores with cost data: 3
   Stores missing cost data: 0
   Sample store with data: Wasilla (CZ: 7)

🏝️  Hawaii (HI) Stores:
   Total stores: 10
   Climate zones: {'1A': 10}
   Stores with cost data: 10
   Stores missing cost data: 0
   Sample store with data: Oahu-Honolulu Salt Lake (CZ: 1A)

📊 States with missing cost data:
   ✅ All states have complete cost data!



In [7]:
# =============================================================================
# DIAGNOSTIC: Why is Hawaii (1A) not getting cost data?
# =============================================================================

print("\n" + "="*80)
print("🔍 INVESTIGATING: Why Hawaii (Climate Zone 1A) has no cost data")
print("="*80)

# Check if Miami is in cz_to_city mapping
print(f"\nClimate zone to city mapping:")
print(f"   Climate Zone '1A' maps to: {cz_to_city.get('1A', 'NOT FOUND')}")

# Check if Miami simulation data exists
if 'Miami' in city_region_state:
    print(f"\n✅ Miami IS in city_region_state dictionary")
    print(f"   State: {city_region_state['Miami']['state']}")
    print(f"   Climate Zone: {city_region_state['Miami']['climate_zone']}")
    print(f"   Cambium Region: {city_region_state['Miami']['cambium_region']}")
else:
    print(f"\n❌ Miami NOT FOUND in city_region_state")

# Check utility rates for HI
print(f"\nUtility rates check:")
hi_rates = df_bills[df_bills['LOCATION STATE/PROVINCE'] == 'HI']
if len(hi_rates) > 0:
    print(f"   ✅ Hawaii utility rates FOUND")
    print(f"      Electric Consumption Rate: ${hi_rates.iloc[0]['Electric Consumption Rate ($/kWh)']:.4f}/kWh")
    print(f"      Electric Demand Rate: ${hi_rates.iloc[0]['Electric Demand Rate ($/kW)']:.2f}/kW")
    print(f"      Gas Rate: ${hi_rates.iloc[0]['Gas Rate ($/Therm)']:.2f}/Therm")
else:
    print(f"   ❌ Hawaii utility rates NOT FOUND - This is likely the problem!")

# Also check Florida for comparison
print(f"\nFlorida (also has missing data) utility rates check:")
fl_rates = df_bills[df_bills['LOCATION STATE/PROVINCE'] == 'FL']
if len(fl_rates) > 0:
    print(f"   ✅ Florida utility rates FOUND")
    print(f"      Electric Consumption Rate: ${fl_rates.iloc[0]['Electric Consumption Rate ($/kWh)']:.4f}/kWh")
    print(f"      Electric Demand Rate: ${fl_rates.iloc[0]['Electric Demand Rate ($/kW)']:.2f}/kW")
    print(f"      Gas Rate: ${fl_rates.iloc[0]['Gas Rate ($/Therm)']:.2f}/Therm")
else:
    print(f"   ❌ Florida utility rates NOT FOUND")

# Show all states with utility rates
print(f"\nAll states with utility rates in df_bills:")
print(f"   {sorted(df_bills['LOCATION STATE/PROVINCE'].unique())}")

print("\n" + "="*80)


🔍 INVESTIGATING: Why Hawaii (Climate Zone 1A) has no cost data

Climate zone to city mapping:
   Climate Zone '1A' maps to: Miami

✅ Miami IS in city_region_state dictionary
   State: FL
   Climate Zone: 1A
   Cambium Region: FRCCc

Utility rates check:
   ✅ Hawaii utility rates FOUND
      Electric Consumption Rate: $0.2594/kWh
      Electric Demand Rate: $26.50/kW
      Gas Rate: $nan/Therm

Florida (also has missing data) utility rates check:
   ✅ Florida utility rates FOUND
      Electric Consumption Rate: $0.1000/kWh
      Electric Demand Rate: $0.71/kW
      Gas Rate: $nan/Therm

All states with utility rates in df_bills:
   ['AK', 'AL', 'AR', 'AZ', 'CA', 'CO', 'CT', 'DC', 'DE', 'FL', 'GA', 'HI', 'IA', 'ID', 'IL', 'IN', 'KS', 'KY', 'LA', 'MA', 'MD', 'ME', 'MI', 'MN', 'MO', 'MS', 'MT', 'NC', 'ND', 'NE', 'NH', 'NJ', 'NM', 'NV', 'NY', 'OH', 'OK', 'OR', 'PA', 'RI', 'SC', 'SD', 'TN', 'TX', 'UT', 'VA', 'VT', 'WA', 'WI', 'WV', 'WY']



### 🔧 Fix Applied: Gas Rate NaN Handling

**Problem identified:** Hawaii and Florida have `NaN` gas rates, causing their total costs to be `NaN`.

**Solution:** Modified the `calculate_store_energy_costs` function to treat `NaN` gas rates as `$0.00/Therm`.

**To apply the fix:**
1. Delete the checkpoint file: `store_costs_checkpoint.csv`
2. Set `RECALCULATE_COSTS = True` 
3. Re-run cell 8 to recalculate all costs with the fix

This makes sense because HI and FL have minimal/no natural gas infrastructure and rely on electricity.

## Plots

In [8]:
# =============================================================================
# FINAL STEP: fix missing totals/savings and create savings boxplots
# =============================================================================

import os
from pathlib import Path

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# -------------------------------------------------------------------------
# 1) Work on a copy of df_stores_with_costs and ensure climate-zone numeric
# -------------------------------------------------------------------------
df = df_stores_with_costs.copy()

# Extract climate zone number (ignore letter suffix, e.g., "5A" -> 5)
df["Climate_Zone_Number"] = (
    df["climate zone"].astype(str).str.extract(r"(\d+)")[0]
)

# Paths for saving results
OUTPUT_CSV = Path("Target_Stores_Energy_Cost_Analysis.csv")
PLOTS_DIR = Path("plots")
PLOTS_DIR.mkdir(exist_ok=True)

# Optional: small backup of the previous CSV if it already exists
if OUTPUT_CSV.exists():
    backup_path = OUTPUT_CSV.with_name(
        OUTPUT_CSV.stem + "_backup" + OUTPUT_CSV.suffix
    )
    df_stores_with_costs.to_csv(backup_path, index=False)
    print(f"💾 Backup of previous CSV saved to: {backup_path}")

# -------------------------------------------------------------------------
# 2) Recalculate Total_Energy_Cost and Savings_vs_Baseline_Percent
#    (inline version of fix_missing_totals.py)
# -------------------------------------------------------------------------
print("🔧 Recalculating total energy costs and savings percentages...")

scenarios = ["Baseline", "NREL_Electric", "NREL_DualFuel",
             "HPC_Electric", "HPC_DualFuel"]

baseline_cost_col = "Baseline_Total_Energy_Cost"

# Rebuild total energy cost from components, treating NaN gas cost as 0
for scenario in scenarios:
    consumption_col = f"{scenario}_Electric_Consumption_Cost"
    demand_col      = f"{scenario}_Electric_Demand_Cost"
    gas_col         = f"{scenario}_Gas_Cost"
    total_col       = f"{scenario}_Total_Energy_Cost"

    missing_cols = [c for c in [consumption_col, demand_col, gas_col]
                    if c not in df.columns]
    if missing_cols:
        print(f"  ⚠️ Skipping {scenario}: missing columns {missing_cols}")
        continue

    df[total_col] = (
        df[consumption_col].fillna(0.0)
        + df[demand_col].fillna(0.0)
        + df[gas_col].fillna(0.0)
    )

    # If *all* components are NaN, keep total as NaN
    all_nan_mask = (
        df[consumption_col].isna()
        & df[demand_col].isna()
        & df[gas_col].isna()
    )
    df.loc[all_nan_mask, total_col] = pd.NA

    print(f"  ✅ {scenario}: recalculated {total_col}")

# Recalculate savings vs baseline (%)
for scenario in scenarios[1:]:  # skip Baseline
    scenario_cost_col = f"{scenario}_Total_Energy_Cost"
    savings_col       = f"{scenario}_Savings_vs_Baseline_Percent"

    if scenario_cost_col not in df.columns or baseline_cost_col not in df.columns:
        print(f"  ⚠️ Skipping savings for {scenario}: missing cost columns")
        continue

    df[savings_col] = (
        (df[baseline_cost_col] - df[scenario_cost_col])
        / df[baseline_cost_col]
        * 100.0
    )

    # Where baseline is NaN or zero, savings is undefined
    df.loc[df[baseline_cost_col].isna() | (df[baseline_cost_col] == 0),
           savings_col] = pd.NA

    print(f"  ✅ {scenario}: recalculated {savings_col}")

# Save updated CSV
df.to_csv(OUTPUT_CSV, index=False)
print(f"\n💾 Updated analysis CSV saved to: {OUTPUT_CSV}")

# -------------------------------------------------------------------------
# 3) Boxplots of savings by climate zone (inline version of
#    plot_savings_by_climate_zone.py)
# -------------------------------------------------------------------------
print("\n📈 Creating savings boxplots by climate zone...")

df_with_costs = df[df[baseline_cost_col].notna()].copy()

plot_scenarios = ["NREL_Electric", "NREL_DualFuel",
                  "HPC_Electric", "HPC_DualFuel"]

scenario_labels = {
    "NREL_Electric": "NREL Lab Data<br>Electric Backup",
    "NREL_DualFuel": "NREL Lab Data<br>Dual Fuel",
    "HPC_Electric":  "HP Challenge<br>Electric Backup",
    "HPC_DualFuel":  "HP Challenge<br>Dual Fuel",
}

baseline_label = "Gas RTU"

climate_zones = sorted(
    df_with_costs["Climate_Zone_Number"].dropna().unique(),
    key=lambda x: int(x)
)

fig_cz = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[scenario_labels[s] for s in plot_scenarios],
    vertical_spacing=0.15,
    horizontal_spacing=0.12,
)

# Colors for climate zones
cz_colors = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728",
    "#9467bd", "#8c564b", "#e377c2",
]

positions = {
    "NREL_Electric": (1, 1),
    "NREL_DualFuel": (1, 2),
    "HPC_Electric":  (2, 1),
    "HPC_DualFuel":  (2, 2),
}

for scenario in plot_scenarios:
    savings_col = f"{scenario}_Savings_vs_Baseline_Percent"
    row, col = positions[scenario]

    for i, zone in enumerate(climate_zones):
        zone_data = df_with_costs.loc[
            df_with_costs["Climate_Zone_Number"] == zone, savings_col
        ].dropna()

        if len(zone_data) == 0:
            continue

        fig_cz.add_trace(
            go.Box(
                y=zone_data,
                name=f"Zone {zone}",
                marker_color=cz_colors[i % len(cz_colors)],
                boxmean="sd",
                showlegend=False,
            ),
            row=row,
            col=col,
        )

# Global y-axis range
all_savings = []
for scenario in plot_scenarios:
    colname = f"{scenario}_Savings_vs_Baseline_Percent"
    if colname in df_with_costs.columns:
        all_savings.extend(df_with_costs[colname].dropna().tolist())

if all_savings:
    y_min = min(all_savings)
    y_max = max(all_savings)
    y_range = [y_min - 2, y_max + 2]
else:
    y_range = [-10, 10]  # fallback

for i in range(1, 5):
    fig_cz.update_yaxes(
        title_text="Savings (%)",
        gridcolor="lightgray",
        zeroline=True,
        zerolinecolor="black",
        zerolinewidth=1,
        range=y_range,
        row=(i - 1) // 2 + 1,
        col=(i - 1) % 2 + 1,
    )
    fig_cz.update_xaxes(
        title_text="Climate Zone",
        row=(i - 1) // 2 + 1,
        col=(i - 1) % 2 + 1,
    )

fig_cz.update_layout(
    title=dict(
        text=f"Energy Cost Savings vs {baseline_label} by Climate Zone",
        font=dict(size=20),
        x=0.5,
        xanchor="center",
    ),
    height=800,
    width=1200,
    template="plotly_white",
    showlegend=False,
)

cz_html = PLOTS_DIR / "savings_by_climate_zone.html"
cz_jpeg = PLOTS_DIR / "savings_by_climate_zone.jpeg"

fig_cz.write_html(str(cz_html))
print(f"📊 Climate-zone HTML plot saved to: {cz_html}")

try:
    fig_cz.write_image(str(cz_jpeg), format="jpeg", scale=2, width=1200, height=800)
    print(f"📊 Climate-zone JPEG plot saved to: {cz_jpeg}")
except Exception as e:
    print(f"⚠️ Could not save climate-zone JPEG (kaleido may not be installed): {e}")

fig_cz.show()

# -------------------------------------------------------------------------
# 4) Boxplots of savings by state (inline version of
#    plot_savings_by_state.py)
# -------------------------------------------------------------------------
print("\n📈 Creating savings boxplots by state...")

state_counts = df_with_costs["State"].value_counts()
states_to_plot = sorted(state_counts[state_counts >= 5].index.tolist())
print(f"States with ≥5 stores: {len(states_to_plot)}")

fig_state = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[scenario_labels[s] for s in plot_scenarios],
    vertical_spacing=0.15,
    horizontal_spacing=0.12,
)

colors_state = px.colors.qualitative.Dark24 + px.colors.qualitative.Light24
positions_state = positions  # same mapping

for scenario in plot_scenarios:
    savings_col = f"{scenario}_Savings_vs_Baseline_Percent"
    row, col = positions_state[scenario]

    for i, state in enumerate(states_to_plot):
        state_data = df_with_costs.loc[
            df_with_costs["State"] == state, savings_col
        ].dropna()

        if len(state_data) == 0:
            continue

        fig_state.add_trace(
            go.Box(
                y=state_data,
                name=state,
                marker_color=colors_state[i % len(colors_state)],
                boxmean="sd",
                showlegend=False,
            ),
            row=row,
            col=col,
        )

# Same y-range as climate plots
for i in range(1, 5):
    fig_state.update_yaxes(
        title_text="Savings (%)",
        gridcolor="lightgray",
        zeroline=True,
        zerolinecolor="black",
        zerolinewidth=1,
        range=y_range,
        row=(i - 1) // 2 + 1,
        col=(i - 1) % 2 + 1,
    )
    fig_state.update_xaxes(
        title_text="State",
        tickangle=-45,
        row=(i - 1) // 2 + 1,
        col=(i - 1) % 2 + 1,
    )

fig_state.update_layout(
    title=dict(
        text=f"Energy Cost Savings vs {baseline_label} by State",
        font=dict(size=20),
        x=0.5,
        xanchor="center",
    ),
    height=800,
    width=1400,
    template="plotly_white",
    showlegend=False,
)

state_html = PLOTS_DIR / "savings_by_state.html"
state_jpeg = PLOTS_DIR / "savings_by_state.jpeg"

fig_state.write_html(str(state_html))
print(f"📊 State-level HTML plot saved to: {state_html}")

try:
    fig_state.write_image(str(state_jpeg), format="jpeg", scale=2, width=1400, height=800)
    print(f"📊 State-level JPEG plot saved to: {state_jpeg}")
except Exception as e:
    print(f"⚠️ Could not save state-level JPEG (kaleido may not be installed): {e}")

fig_state.show()

print("\n✅ Final fixes and plots complete.")


💾 Backup of previous CSV saved to: Target_Stores_Energy_Cost_Analysis_backup.csv
🔧 Recalculating total energy costs and savings percentages...
  ✅ Baseline: recalculated Baseline_Total_Energy_Cost
  ✅ NREL_Electric: recalculated NREL_Electric_Total_Energy_Cost
  ✅ NREL_DualFuel: recalculated NREL_DualFuel_Total_Energy_Cost
  ✅ HPC_Electric: recalculated HPC_Electric_Total_Energy_Cost
  ✅ HPC_DualFuel: recalculated HPC_DualFuel_Total_Energy_Cost
  ✅ NREL_Electric: recalculated NREL_Electric_Savings_vs_Baseline_Percent
  ✅ NREL_DualFuel: recalculated NREL_DualFuel_Savings_vs_Baseline_Percent
  ✅ HPC_Electric: recalculated HPC_Electric_Savings_vs_Baseline_Percent
  ✅ HPC_DualFuel: recalculated HPC_DualFuel_Savings_vs_Baseline_Percent

💾 Updated analysis CSV saved to: Target_Stores_Energy_Cost_Analysis.csv

📈 Creating savings boxplots by climate zone...
📊 Climate-zone HTML plot saved to: plots/savings_by_climate_zone.html
📊 Climate-zone JPEG plot saved to: plots/savings_by_climate_zone.jpe


📈 Creating savings boxplots by state...
States with ≥5 stores: 46
📊 State-level HTML plot saved to: plots/savings_by_state.html
📊 State-level JPEG plot saved to: plots/savings_by_state.jpeg



✅ Final fixes and plots complete.


In [9]:
# =============================================================================
# FINAL STEP: fix missing totals/savings and create savings maps
# =============================================================================

import os
from pathlib import Path

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# -------------------------------------------------------------------------
# 1) Work on a copy of df_stores_with_costs and ensure climate-zone numeric
# -------------------------------------------------------------------------
df = df_stores_with_costs.copy()

# Extract climate zone number (ignore letter suffix, e.g., "5A" -> 5)
df["Climate_Zone_Number"] = (
    df["climate zone"].astype(str).str.extract(r"(\d+)")[0]
)

# Paths for saving results
OUTPUT_CSV = Path("Target_Stores_Energy_Cost_Analysis.csv")
PLOTS_DIR = Path("plots")
PLOTS_DIR.mkdir(exist_ok=True)

# Optional: small backup of the previous CSV if it already exists
if OUTPUT_CSV.exists():
    backup_path = OUTPUT_CSV.with_name(
        OUTPUT_CSV.stem + "_backup" + OUTPUT_CSV.suffix
    )
    df_stores_with_costs.to_csv(backup_path, index=False)
    print(f"💾 Backup of previous CSV saved to: {backup_path}")

# -------------------------------------------------------------------------
# 2) Recalculate Total_Energy_Cost and Savings_vs_Baseline_Percent
# -------------------------------------------------------------------------
print("🔧 Recalculating total energy costs and savings percentages...")

scenarios = ["Baseline", "NREL_Electric", "NREL_DualFuel",
             "HPC_Electric", "HPC_DualFuel"]

baseline_cost_col = "Baseline_Total_Energy_Cost"

# Rebuild total energy cost from components, treating NaN gas cost as 0
for scenario in scenarios:
    consumption_col = f"{scenario}_Electric_Consumption_Cost"
    demand_col      = f"{scenario}_Electric_Demand_Cost"
    gas_col         = f"{scenario}_Gas_Cost"
    total_col       = f"{scenario}_Total_Energy_Cost"

    missing_cols = [c for c in [consumption_col, demand_col, gas_col]
                    if c not in df.columns]
    if missing_cols:
        print(f"  ⚠️ Skipping {scenario}: missing columns {missing_cols}")
        continue

    df[total_col] = (
        df[consumption_col].fillna(0.0)
        + df[demand_col].fillna(0.0)
        + df[gas_col].fillna(0.0)
    )

    # If *all* components are NaN, keep total as NaN
    all_nan_mask = (
        df[consumption_col].isna()
        & df[demand_col].isna()
        & df[gas_col].isna()
    )
    df.loc[all_nan_mask, total_col] = pd.NA

    print(f"  ✅ {scenario}: recalculated {total_col}")

# Recalculate savings vs baseline (%)
for scenario in scenarios[1:]:  # skip Baseline
    scenario_cost_col = f"{scenario}_Total_Energy_Cost"
    savings_col       = f"{scenario}_Savings_vs_Baseline_Percent"

    if scenario_cost_col not in df.columns or baseline_cost_col not in df.columns:
        print(f"  ⚠️ Skipping savings for {scenario}: missing cost columns")
        continue

    df[savings_col] = (
        (df[baseline_cost_col] - df[scenario_cost_col])
        / df[baseline_cost_col]
        * 100.0
    )

    # Where baseline is NaN or zero, savings is undefined
    df.loc[df[baseline_cost_col].isna() | (df[baseline_cost_col] == 0),
           savings_col] = pd.NA

    print(f"  ✅ {scenario}: recalculated {savings_col}")

# Save updated CSV
df.to_csv(OUTPUT_CSV, index=False)
print(f"\n💾 Updated analysis CSV saved to: {OUTPUT_CSV}")

# -------------------------------------------------------------------------
# 3) Create state-level savings choropleth maps (4 scenarios)
# -------------------------------------------------------------------------
print("\n📈 Creating state-level savings choropleth maps...")

# Compute state mean savings for the 4 heat pump scenarios
map_scenarios = ["NREL_Electric", "NREL_DualFuel", "HPC_Electric", "HPC_DualFuel"]

scenario_labels = {
    "NREL_Electric": "NREL Lab Data<br>Electric Backup",
    "NREL_DualFuel": "NREL Lab Data<br>Dual Fuel",
    "HPC_Electric":  "HP Challenge<br>Electric Backup",
    "HPC_DualFuel":  "HP Challenge<br>Dual Fuel",
}

baseline_label = "Gas RTU"

state_savings = {}
for scenario in map_scenarios:
    savings_col = f"{scenario}_Savings_vs_Baseline_Percent"
    # Group by state, calculate mean savings
    mean_by_state = df_stores_with_costs.groupby("State")[savings_col].mean()
    state_savings[scenario] = mean_by_state

# Collect all values to determine color scale range
values = []
for scenario_data in state_savings.values():
    values.extend(scenario_data.dropna().values.tolist())

if len(values) == 0:
    zmin, zmax = -5, 5
else:
    # use 90th percentile of absolute values to clip outliers
    max_abs = np.percentile(np.abs(values), 90)
    if max_abs <= 0:
        max_abs = 1e-3  # fallback to avoid zero range
    zmin = -max_abs
    zmax =  max_abs

    # Approximate state center coordinates (latitude, longitude)
    state_coords = {
        'AL': (32.806671, -86.791130), 'AK': (61.370716, -152.404419),
        'AZ': (33.729759, -111.431221), 'AR': (34.969704, -92.373123),
        'CA': (36.116203, -119.681564), 'CO': (39.059811, -105.311104),
        'CT': (41.597782, -72.755371), 'DE': (39.318523, -75.507141),
        'FL': (27.766279, -81.686783), 'GA': (33.040619, -83.643074),
        'HI': (21.094318, -157.498337), 'ID': (44.240459, -114.478828),
        'IL': (40.349457, -88.986137), 'IN': (39.849426, -86.258278),
        'IA': (42.011539, -93.210526), 'KS': (38.526600, -96.726486),
        'KY': (37.668140, -84.670067), 'LA': (31.169546, -91.867805),
        'ME': (44.693947, -69.381927), 'MD': (39.063946, -76.802101),
        'MA': (42.230171, -71.530106), 'MI': (43.326618, -84.536095),
        'MN': (45.694454, -93.900192), 'MS': (32.741646, -89.678696),
        'MO': (38.456085, -92.288368), 'MT': (46.921925, -110.454353),
        'NE': (41.125370, -98.268082), 'NV': (38.313515, -117.055374),
        'NH': (43.452492, -71.563896), 'NJ': (40.298904, -74.521011),
        'NM': (34.840515, -106.248482), 'NY': (42.165726, -74.948051),
        'NC': (35.630066, -79.806419), 'ND': (47.528912, -99.784012),
        'OH': (40.388783, -82.764915), 'OK': (35.565342, -96.928917),
        'OR': (44.572021, -122.070938), 'PA': (40.590752, -77.209755),
        'RI': (41.680893, -71.511780), 'SC': (33.856892, -80.945007),
        'SD': (44.299782, -99.438828), 'TN': (35.747845, -86.692345),
        'TX': (31.054487, -97.563461), 'UT': (40.150032, -111.862434),
        'VT': (44.045876, -72.710686), 'VA': (37.769337, -78.169968),
        'WA': (47.400902, -121.490494), 'WV': (38.491226, -80.954453),
        'WI': (44.268543, -89.616508), 'WY': (42.755966, -107.302490)
    }

    # Create a 2x2 subplot for 4 scenarios with maximum spacing
    fig_maps = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=[scenario_labels[s].replace("<br>", " ") for s in map_scenarios],
        specs=[[{"type": "choropleth"}, {"type": "choropleth"}],
               [{"type": "choropleth"}, {"type": "choropleth"}]],
        horizontal_spacing=0.01,  # Minimize horizontal spacing
        vertical_spacing=0.08,    # Minimize vertical spacing
    )

    for idx, scenario in enumerate(map_scenarios):
        state_mean = state_savings[scenario]
        row = idx // 2 + 1
        col = idx % 2 + 1

        fig_maps.add_choropleth(
            locations=state_mean.index,
            z=state_mean.values,
            locationmode="USA-states",
            coloraxis="coloraxis",
            row=row, col=col,
        )
        
        # Add text annotations for each state showing savings percentage
        for state_code in state_mean.index:
            if state_code in state_coords:
                lat, lon = state_coords[state_code]
                savings_value = state_mean[state_code]
                
                fig_maps.add_scattergeo(
                    lon=[lon],
                    lat=[lat],
                    text=[f"{savings_value:.1f}"],
                    mode='text',
                    textfont=dict(size=11, color='black', family='Arial Black'),
                    showlegend=False,
                    row=row, col=col,
                )

    fig_maps.update_layout(
        title=dict(
            text=f"Energy Cost Savings vs {baseline_label} by State (%)",
            font=dict(size=20),
            x=0.5, xanchor="center",
        ),
        template="plotly_white",
        height=1400,
        width=2000,
        margin=dict(t=80, b=30, l=30, r=30),
        coloraxis=dict(
            # diverging: red (neg), white (0), green (pos)
            colorscale=[
                [0.0,  "#8b0000"],  # dark red
                [0.25, "#ff9999"],  # light red
                [0.5,  "#ffffff"],  # white at 0%
                [0.75, "#99ff99"],  # light green
                [1.0,  "#006400"],  # dark green
            ],
            cmin=zmin,
            cmax=zmax,
            colorbar_title="Savings (%)",
        ),
        geo=dict(scope="usa", projection_type="albers usa", showlakes=False),
        geo2=dict(scope="usa", projection_type="albers usa", showlakes=False),
        geo3=dict(scope="usa", projection_type="albers usa", showlakes=False),
        geo4=dict(scope="usa", projection_type="albers usa", showlakes=False),
    )

    state_html = PLOTS_DIR / "savings_by_state_maps_combined.html"
    state_jpeg = PLOTS_DIR / "savings_by_state_maps_combined.jpeg"
    fig_maps.write_html(str(state_html))
    fig_maps.write_image(str(state_jpeg), width=2000, height=1400, scale=3)
    fig_maps.show()

    print("\n✅ Combined state-level choropleth maps created (4 scenarios, all states included with labels).")
    print(f"   Saved to: {state_html}")
    print(f"   Saved to: {state_jpeg}")


💾 Backup of previous CSV saved to: Target_Stores_Energy_Cost_Analysis_backup.csv
🔧 Recalculating total energy costs and savings percentages...
  ✅ Baseline: recalculated Baseline_Total_Energy_Cost
  ✅ NREL_Electric: recalculated NREL_Electric_Total_Energy_Cost
  ✅ NREL_DualFuel: recalculated NREL_DualFuel_Total_Energy_Cost
  ✅ HPC_Electric: recalculated HPC_Electric_Total_Energy_Cost
  ✅ HPC_DualFuel: recalculated HPC_DualFuel_Total_Energy_Cost
  ✅ NREL_Electric: recalculated NREL_Electric_Savings_vs_Baseline_Percent
  ✅ NREL_DualFuel: recalculated NREL_DualFuel_Savings_vs_Baseline_Percent
  ✅ HPC_Electric: recalculated HPC_Electric_Savings_vs_Baseline_Percent
  ✅ HPC_DualFuel: recalculated HPC_DualFuel_Savings_vs_Baseline_Percent

💾 Updated analysis CSV saved to: Target_Stores_Energy_Cost_Analysis.csv

📈 Creating state-level savings choropleth maps...



✅ Combined state-level choropleth maps created (4 scenarios, all states included with labels).
   Saved to: plots/savings_by_state_maps_combined.html
   Saved to: plots/savings_by_state_maps_combined.jpeg


In [10]:
# =============================================================================
# GHG EMISSIONS ANALYSIS: Calculate CO2 savings and create map visualizations
# =============================================================================

print("\n🌍 Calculating GHG emissions savings by location...")

# Map scenario names from city_scenario_emissions to our column naming convention
scenario_mapping = {
    'Baseline': 'Baseline',
    '2 Speed - Lab Data - Electric Backup': 'NREL_Electric',
    '2 Speed - Lab Data - Gas Backup': 'NREL_DualFuel',
    'HPC curves - Electric Backup': 'HPC_Electric',
    'HP Challenge curves - Gas Backup': 'HPC_DualFuel'
}

# Create a mapping from city to climate zone for matching with stores
city_to_cz = {city: data['climate_zone'] for city, data in city_scenario_emissions.items()}

# Initialize columns for GHG emissions in df_stores_with_costs
for scenario_key in ['Baseline', 'NREL_Electric', 'NREL_DualFuel', 'HPC_Electric', 'HPC_DualFuel']:
    df_stores_with_costs[f'{scenario_key}_GHG_Emissions_kg'] = np.nan
    if scenario_key != 'Baseline':
        df_stores_with_costs[f'{scenario_key}_GHG_Savings_vs_Baseline_kg'] = np.nan
        df_stores_with_costs[f'{scenario_key}_GHG_Savings_vs_Baseline_Percent'] = np.nan

# For each store, find matching city data and scale emissions by area
# We'll use the energy data columns to determine which stores have valid data
stores_with_energy = df_stores_with_costs['Baseline_Electricity_kWh'].notna()

for idx, row in df_stores_with_costs[stores_with_energy].iterrows():
    store_cz = str(row['climate zone'])
    
    # Get store area from baseline electricity if available, otherwise skip
    # We'll use a ratio approach: scale emissions by electricity ratio
    baseline_elec_kwh = row['Baseline_Electricity_kWh']
    
    if pd.isna(baseline_elec_kwh) or baseline_elec_kwh == 0:
        continue
    
    # Find matching city by climate zone
    matching_city = None
    for city, cz in city_to_cz.items():
        if cz == store_cz:
            matching_city = city
            break
    
    if matching_city is None:
        continue
    
    city_data = city_scenario_emissions[matching_city]
    
    # For scaling, we need to get the model's baseline electricity from city_scenario_emissions
    # The emissions are already calculated for the model size, so we scale by energy ratio
    
    # Calculate scaled emissions for each scenario
    for orig_scenario, our_scenario in scenario_mapping.items():
        if orig_scenario in city_data['scenarios']:
            scenario_data = city_data['scenarios'][orig_scenario]
            total_ghg_kg = scenario_data['total_ghg_kg']
            
            if total_ghg_kg is not None:
                # Get the store's actual energy for this scenario
                store_elec_col = f'{our_scenario}_Electricity_kWh'
                
                if store_elec_col in df_stores_with_costs.columns:
                    store_elec = row[store_elec_col]
                    
                    # Scale emissions proportionally to electricity usage
                    # This is a simplification but reasonable for similar building types
                    if not pd.isna(store_elec) and not pd.isna(baseline_elec_kwh) and baseline_elec_kwh > 0:
                        # Use energy ratio relative to baseline
                        scale_factor = store_elec / baseline_elec_kwh
                        
                        # For baseline, just use the baseline electricity ratio
                        if our_scenario == 'Baseline':
                            baseline_ghg_model = city_data['scenarios']['Baseline']['total_ghg_kg']
                            if baseline_ghg_model and baseline_ghg_model > 0:
                                scaled_ghg = total_ghg_kg * (baseline_elec_kwh / baseline_elec_kwh)  # This is just total_ghg_kg
                                # Actually, let's use MODEL_AREA for proper scaling
                                # Emissions per sqft from model, scaled to store
                                if 'Square Footage' in df_stores_with_costs.columns:
                                    store_sqft = row['Square Footage']
                                else:
                                    # Use electricity as proxy if square footage not available
                                    store_sqft = None
                                
                                if store_sqft and not pd.isna(store_sqft):
                                    scaled_ghg = total_ghg_kg * (store_sqft / MODEL_AREA_SQFT)
                                else:
                                    # Fall back to electricity ratio
                                    scaled_ghg = total_ghg_kg
                        else:
                            # For other scenarios, scale by store size
                            if 'Square Footage' in df_stores_with_costs.columns:
                                store_sqft = row['Square Footage']
                            else:
                                store_sqft = None
                            
                            if store_sqft and not pd.isna(store_sqft):
                                scaled_ghg = total_ghg_kg * (store_sqft / MODEL_AREA_SQFT)
                            else:
                                scaled_ghg = total_ghg_kg
                        
                        df_stores_with_costs.at[idx, f'{our_scenario}_GHG_Emissions_kg'] = scaled_ghg

# Calculate GHG savings (baseline - scenario)
baseline_ghg_col = 'Baseline_GHG_Emissions_kg'
for scenario in ['NREL_Electric', 'NREL_DualFuel', 'HPC_Electric', 'HPC_DualFuel']:
    ghg_col = f'{scenario}_GHG_Emissions_kg'
    savings_kg_col = f'{scenario}_GHG_Savings_vs_Baseline_kg'
    savings_pct_col = f'{scenario}_GHG_Savings_vs_Baseline_Percent'
    
    df_stores_with_costs[savings_kg_col] = (
        df_stores_with_costs[baseline_ghg_col] - df_stores_with_costs[ghg_col]
    )
    
    df_stores_with_costs[savings_pct_col] = (
        (df_stores_with_costs[baseline_ghg_col] - df_stores_with_costs[ghg_col]) 
        / df_stores_with_costs[baseline_ghg_col] 
        * 100.0
    )
    
    # Set NaN where baseline is missing or zero
    df_stores_with_costs.loc[
        df_stores_with_costs[baseline_ghg_col].isna() | (df_stores_with_costs[baseline_ghg_col] == 0),
        [savings_kg_col, savings_pct_col]
    ] = pd.NA

print(f"✅ GHG emissions calculated for {df_stores_with_costs[baseline_ghg_col].notna().sum()} stores")

# Filter stores with valid GHG data
df_stores_ghg = df_stores_with_costs[df_stores_with_costs[baseline_ghg_col].notna()].copy()

# -------------------------------------------------------------------------
# 1) GHG Savings Maps (4 scenarios)
# -------------------------------------------------------------------------
print("\n🗺️ Creating GHG savings choropleth maps by state...")

states_to_use_ghg = df_stores_ghg["State"].dropna().unique().tolist()
map_scenarios_ghg = ["NREL_Electric", "NREL_DualFuel", "HPC_Electric", "HPC_DualFuel"]

state_ghg_savings = {}
all_ghg_state_values = []

for scenario in map_scenarios_ghg:
    savings_pct_col = f"{scenario}_GHG_Savings_vs_Baseline_Percent"
    
    state_mean = (
        df_stores_ghg[df_stores_ghg["State"].isin(states_to_use_ghg)]
        .groupby("State")[savings_pct_col]
        .mean()
        .dropna()
    )
    
    state_ghg_savings[scenario] = state_mean
    all_ghg_state_values.extend(state_mean.values.tolist())

# Calculate color range
values_ghg = np.array(all_ghg_state_values)
if values_ghg.size == 0:
    print("⚠️ No state-level GHG savings data available to plot.")
else:
    max_abs_ghg = np.percentile(np.abs(values_ghg), 90)
    if max_abs_ghg <= 0:
        max_abs_ghg = 1e-3
    zmin_ghg = -max_abs_ghg
    zmax_ghg = max_abs_ghg

    # State coordinates for labels
    state_coords = {
        'AL': (32.806671, -86.791130), 'AK': (61.370716, -152.404419),
        'AZ': (33.729759, -111.431221), 'AR': (34.969704, -92.373123),
        'CA': (36.116203, -119.681564), 'CO': (39.059811, -105.311104),
        'CT': (41.597782, -72.755371), 'DE': (39.318523, -75.507141),
        'FL': (27.766279, -81.686783), 'GA': (33.040619, -83.643074),
        'HI': (21.094318, -157.498337), 'ID': (44.240459, -114.478828),
        'IL': (40.349457, -88.986137), 'IN': (39.849426, -86.258278),
        'IA': (42.011539, -93.210526), 'KS': (38.526600, -96.726486),
        'KY': (37.668140, -84.670067), 'LA': (31.169546, -91.867805),
        'ME': (44.693947, -69.381927), 'MD': (39.063946, -76.802101),
        'MA': (42.230171, -71.530106), 'MI': (43.326618, -84.536095),
        'MN': (45.694454, -93.900192), 'MS': (32.741646, -89.678696),
        'MO': (38.456085, -92.288368), 'MT': (46.921925, -110.454353),
        'NE': (41.125370, -98.268082), 'NV': (38.313515, -117.055374),
        'NH': (43.452492, -71.563896), 'NJ': (40.298904, -74.521011),
        'NM': (34.840515, -106.248482), 'NY': (42.165726, -74.948051),
        'NC': (35.630066, -79.806419), 'ND': (47.528912, -99.784012),
        'OH': (40.388783, -82.764915), 'OK': (35.565342, -96.928917),
        'OR': (44.572021, -122.070938), 'PA': (40.590752, -77.209755),
        'RI': (41.680893, -71.511780), 'SC': (33.856892, -80.945007),
        'SD': (44.299782, -99.438828), 'TN': (35.747845, -86.692345),
        'TX': (31.054487, -97.563461), 'UT': (40.150032, -111.862434),
        'VT': (44.045876, -72.710686), 'VA': (37.769337, -78.169968),
        'WA': (47.400902, -121.490494), 'WV': (38.491226, -80.954453),
        'WI': (44.268543, -89.616508), 'WY': (42.755966, -107.302490)
    }

    fig_ghg = make_subplots(
        rows=2, cols=2,
        subplot_titles=[scenario_labels[s].replace("<br>", " ") for s in map_scenarios_ghg],
        specs=[[{"type": "choropleth"}, {"type": "choropleth"}],
               [{"type": "choropleth"}, {"type": "choropleth"}]],
        horizontal_spacing=0.01,
        vertical_spacing=0.08,
    )

    for idx, scenario in enumerate(map_scenarios_ghg):
        state_mean = state_ghg_savings[scenario]
        row = idx // 2 + 1
        col = idx % 2 + 1

        fig_ghg.add_choropleth(
            locations=state_mean.index,
            z=state_mean.values,
            locationmode="USA-states",
            coloraxis="coloraxis",
            row=row, col=col,
        )
        
        # Add text labels
        for state_code in state_mean.index:
            if state_code in state_coords:
                lat, lon = state_coords[state_code]
                savings_value = state_mean[state_code]
                
                fig_ghg.add_scattergeo(
                    lon=[lon], lat=[lat],
                    text=[f"{savings_value:.1f}"],
                    mode='text',
                    textfont=dict(size=11, color='black', family='Arial Black'),
                    showlegend=False,
                    row=row, col=col,
                )

    fig_ghg.update_layout(
        title=dict(
            text=f"GHG Emissions Savings vs {baseline_label} by State (%)",
            font=dict(size=22),
            x=0.5, xanchor="center",
        ),
        template="plotly_white",
        height=1200, width=1800,
        coloraxis=dict(
            colorscale=[
                [0.0,  "#8b0000"], [0.25, "#ff9999"],
                [0.5,  "#ffffff"], [0.75, "#99ff99"],
                [1.0,  "#006400"],
            ],
            cmin=zmin_ghg, cmax=zmax_ghg,
            colorbar_title="GHG Savings (%)",
        ),
        geo=dict(scope="usa", projection_type="albers usa", showlakes=False),
        geo2=dict(scope="usa", projection_type="albers usa", showlakes=False),
        geo3=dict(scope="usa", projection_type="albers usa", showlakes=False),
        geo4=dict(scope="usa", projection_type="albers usa", showlakes=False),
    )

    ghg_html = PLOTS_DIR / "ghg_savings_by_state.html"
    ghg_jpeg = PLOTS_DIR / "ghg_savings_by_state.jpeg"
    fig_ghg.write_image(str(ghg_jpeg), width=2000, height=1400, scale=3)
    fig_ghg.write_image(str(ghg_jpeg), width=1800, height=1200, scale=3)
    fig_ghg.show()

print("✅ GHG savings maps created")
print(f"   Saved to: {ghg_html}")
print(f"   Saved to: {ghg_jpeg}")

# -------------------------------------------------------------------------
# 2) Combined Cost + GHG Emissions Maps (costs as choropleth, GHG as circles)
# -------------------------------------------------------------------------
print("\n🗺️ Creating combined cost savings + GHG emissions maps...")

# Calculate total GHG emissions (kg) per state for each scenario
state_total_ghg = {}
for scenario in map_scenarios_ghg:
    ghg_col = f"{scenario}_GHG_Emissions_kg"
    
    state_total = (
        df_stores_ghg[df_stores_ghg["State"].isin(states_to_use_ghg)]
        .groupby("State")[ghg_col]
        .sum()
        .dropna()
    )
    
    state_total_ghg[scenario] = state_total

# Get cost savings from previous analysis
state_cost_savings = state_savings  # Already calculated in previous cell

fig_combined = make_subplots(
    rows=2, cols=2,
    subplot_titles=[scenario_labels[s].replace("<br>", " ") for s in map_scenarios_ghg],
    specs=[[{"type": "choropleth"}, {"type": "choropleth"}],
           [{"type": "choropleth"}, {"type": "choropleth"}]],
    horizontal_spacing=0.01,
    vertical_spacing=0.08,
)

# Normalize GHG emissions for circle sizes (use log scale for better visualization)
all_ghg_emissions = []
for scenario in map_scenarios_ghg:
    if scenario in state_total_ghg:
        all_ghg_emissions.extend(state_total_ghg[scenario].values.tolist())

max_ghg = max(all_ghg_emissions) if all_ghg_emissions else 1
min_ghg = min(all_ghg_emissions) if all_ghg_emissions else 0

for idx, scenario in enumerate(map_scenarios_ghg):
    row = idx // 2 + 1
    col = idx % 2 + 1
    
    # Add cost savings choropleth
    if scenario in state_cost_savings:
        state_mean_cost = state_cost_savings[scenario]
        
        fig_combined.add_choropleth(
            locations=state_mean_cost.index,
            z=state_mean_cost.values,
            locationmode="USA-states",
            coloraxis="coloraxis",
            row=row, col=col,
        )
    
    # Add GHG emissions as circles
    if scenario in state_total_ghg:
        state_ghg = state_total_ghg[scenario]
        
        lons = []
        lats = []
        sizes = []
        hover_texts = []
        
        for state_code in state_ghg.index:
            if state_code in state_coords:
                lat, lon = state_coords[state_code]
                ghg_kg = state_ghg[state_code]
                
                # Convert to metric tons for display
                ghg_tons = ghg_kg / 1000
                
                # Size proportional to emissions (with much bigger variance)
                size = 5 + (ghg_kg - min_ghg) / (max_ghg - min_ghg) * 80 if max_ghg > min_ghg else 20
                
                lons.append(lon)
                lats.append(lat)
                sizes.append(size)
                hover_texts.append(f"{state_code}: {ghg_tons:.1f} tons CO2e")
        
        fig_combined.add_scattergeo(
            lon=lons, lat=lats,
            marker=dict(
                size=sizes,
                color='rgba(0, 200, 0, 0.6)',
                line=dict(width=1, color='darkgreen')
            ),
            hovertext=hover_texts,
            hoverinfo='text',
            showlegend=False,
            row=row, col=col,
        )

fig_combined.update_layout(
    title=dict(
        text=f"Cost Savings (%) + Total GHG Emissions (green circles) by State",
        font=dict(size=22),
        x=0.5, xanchor="center",
    ),
    template="plotly_white",
    height=1200, width=1800,
    coloraxis=dict(
        colorscale=[
            [0.0,  "#8b0000"], [0.25, "#ff9999"],
            [0.5,  "#ffffff"], [0.75, "#99ff99"],
            [1.0,  "#006400"],
        ],
        cmin=zmin, cmax=zmax,
        colorbar_title="Cost Savings (%)",
    ),
    geo=dict(scope="usa", projection_type="albers usa", showlakes=False),
    geo2=dict(scope="usa", projection_type="albers usa", showlakes=False),
    geo3=dict(scope="usa", projection_type="albers usa", showlakes=False),
    geo4=dict(scope="usa", projection_type="albers usa", showlakes=False),
)

combined_html = PLOTS_DIR / "cost_savings_ghg_emissions_combined.html"
combined_jpeg = PLOTS_DIR / "cost_savings_ghg_emissions_combined.jpeg"
fig_combined.write_html(str(combined_html))
fig_combined.write_image(str(combined_jpeg), width=1800, height=1200, scale=3)
fig_combined.show()

print("✅ Combined cost + GHG emissions maps created")
print(f"   Saved to: {combined_html}")
print(f"   Saved to: {combined_jpeg}")



🌍 Calculating GHG emissions savings by location...
✅ GHG emissions calculated for 1987 stores

🗺️ Creating GHG savings choropleth maps by state...


✅ GHG savings maps created
   Saved to: plots/ghg_savings_by_state.html
   Saved to: plots/ghg_savings_by_state.jpeg

🗺️ Creating combined cost savings + GHG emissions maps...


✅ Combined cost + GHG emissions maps created
   Saved to: plots/cost_savings_ghg_emissions_combined.html
   Saved to: plots/cost_savings_ghg_emissions_combined.jpeg


In [11]:
# =============================================================================
# BAR PLOTS: Total Costs and GHG Emissions by Scenario
# =============================================================================

print("\n📊 Creating bar plots for total costs and GHG emissions...")

# Calculate totals across all stores for each scenario
scenarios_bar = ["Baseline", "NREL_Electric", "NREL_DualFuel", "HPC_Electric", "HPC_DualFuel"]

# Prepare data for costs
total_costs = []
total_elec_costs = []
total_gas_costs = []
cost_savings_vs_baseline = []

for scenario in scenarios_bar:
    total_cost_col = f"{scenario}_Total_Energy_Cost"
    elec_consumption_col = f"{scenario}_Electric_Consumption_Cost"
    elec_demand_col = f"{scenario}_Electric_Demand_Cost"
    gas_cost_col = f"{scenario}_Gas_Cost"
    
    total_cost = df_stores_with_costs[total_cost_col].sum()
    elec_cost = (df_stores_with_costs[elec_consumption_col].sum() + 
                 df_stores_with_costs[elec_demand_col].sum())
    gas_cost = df_stores_with_costs[gas_cost_col].sum()
    
    total_costs.append(total_cost)
    total_elec_costs.append(elec_cost)
    total_gas_costs.append(gas_cost)
    
    if scenario == "Baseline":
        cost_savings_vs_baseline.append(0)  # Baseline has 0% savings
    else:
        baseline_total = total_costs[0]
        savings_pct = (baseline_total - total_cost) / baseline_total * 100
        cost_savings_vs_baseline.append(savings_pct)

# Prepare data for GHG emissions
total_ghg = []
total_elec_ghg = []
total_gas_ghg = []
ghg_savings_vs_baseline = []

for scenario in scenarios_bar:
    ghg_total_col = f"{scenario}_GHG_Emissions_kg"
    
    if ghg_total_col in df_stores_with_costs.columns:
        total_ghg_val = df_stores_with_costs[ghg_total_col].sum() / 1000  # Convert to metric tons
        total_ghg.append(total_ghg_val)
        
        # For simplicity, we'll just show total (actual breakdown would require more detailed data)
        total_elec_ghg.append(total_ghg_val)  # Placeholder
        total_gas_ghg.append(0)  # Placeholder
        
        if scenario == "Baseline":
            ghg_savings_vs_baseline.append(0)
        else:
            baseline_ghg = total_ghg[0]
            savings_pct = (baseline_ghg - total_ghg_val) / baseline_ghg * 100
            ghg_savings_vs_baseline.append(savings_pct)
    else:
        total_ghg.append(0)
        total_elec_ghg.append(0)
        total_gas_ghg.append(0)
        ghg_savings_vs_baseline.append(0)

# Create labels
scenario_labels_bar = {
    "Baseline": "Baseline<br>(Gas RTU)",
    "NREL_Electric": "NREL Lab Data<br>Electric Backup",
    "NREL_DualFuel": "NREL Lab Data<br>Dual Fuel",
    "HPC_Electric": "HP Challenge<br>Electric Backup",
    "HPC_DualFuel": "HP Challenge<br>Dual Fuel",
}

labels = [scenario_labels_bar[s] for s in scenarios_bar]

# Create figure with 2 subplots
fig_bars = make_subplots(
    rows=2, cols=1,
    subplot_titles=["Total Annual Energy Costs by Scenario", 
                    "Total Annual GHG Emissions by Scenario"],
    vertical_spacing=0.15,
    specs=[[{"type": "bar"}], [{"type": "bar"}]]
)

# Colors for the bars
bar_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

# Top subplot: Costs (in Million $)
for i, scenario in enumerate(scenarios_bar):
    elec_cost_m = total_elec_costs[i] / 1e6
    gas_cost_m = total_gas_costs[i] / 1e6
    
    # Add electricity cost (bottom part)
    fig_bars.add_trace(
        go.Bar(
            x=[labels[i]],
            y=[elec_cost_m],
            name="Electricity Cost" if i == 0 else "",
            marker_color='#4A90E2',
            showlegend=(i == 0),
            legendgroup="elec",
            text=f"${elec_cost_m:.0f}M",
            textposition='inside',
            textfont=dict(size=12, color='white', family='Arial Black'),
            hovertemplate=f"Electricity: ${total_elec_costs[i]:,.0f}<extra></extra>"
        ),
        row=1, col=1
    )
    
    # Add gas cost (top part)
    fig_bars.add_trace(
        go.Bar(
            x=[labels[i]],
            y=[gas_cost_m],
            name="Gas Cost" if i == 0 else "",
            marker_color='#E67E22',
            showlegend=(i == 0),
            legendgroup="gas",
            text=f"${gas_cost_m:.0f}M",
            textposition='inside',
            textfont=dict(size=12, color='white', family='Arial Black'),
            hovertemplate=f"Gas: ${total_gas_costs[i]:,.0f}<extra></extra>"
        ),
        row=1, col=1
    )
    
    # Add savings percentage (1 decimal, skip for baseline)
    if scenario != "Baseline":
        fig_bars.add_annotation(
            x=labels[i],
            y=elec_cost_m + gas_cost_m,
            text=f"{cost_savings_vs_baseline[i]:+.1f}%",
            showarrow=False,
            yshift=20,
            font=dict(size=11, color='green' if cost_savings_vs_baseline[i] > 0 else 'red'),
            row=1, col=1
        )

# Bottom subplot: GHG Emissions (in k Metric tons)
for i, scenario in enumerate(scenarios_bar):
    ghg_k = total_ghg[i] / 1000
    
    # Add GHG bar
    fig_bars.add_trace(
        go.Bar(
            x=[labels[i]],
            y=[ghg_k],
            name="GHG Emissions" if i == 0 else "",
            marker_color='#27AE60',
            showlegend=False,
            text=f"{ghg_k:.0f}k",
            textposition='inside',
            textfont=dict(size=12, color='white', family='Arial Black'),
            hovertemplate=f"Total: {total_ghg[i]:,.0f} tons CO2e<extra></extra>"
        ),
        row=2, col=1
    )
    
    # Add savings percentage (1 decimal, skip for baseline)
    if scenario != "Baseline":
        fig_bars.add_annotation(
            x=labels[i],
            y=ghg_k,
            text=f"{ghg_savings_vs_baseline[i]:+.1f}%",
            showarrow=False,
            yshift=20,
            font=dict(size=11, color='green' if ghg_savings_vs_baseline[i] > 0 else 'red'),
            row=2, col=1
        )

# Update layout
fig_bars.update_layout(
    barmode='stack',
    height=1000,
    width=1200,
    template="plotly_white",
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    ),
    title=dict(
        text="Total Energy Costs and GHG Emissions by Scenario",
        font=dict(size=20),
        x=0.5,
        xanchor="center",
    ),
)

# Update axes
fig_bars.update_yaxes(title_text="Total Cost (Million $)", row=1, col=1)
fig_bars.update_yaxes(title_text="Total GHG (k metric tons CO2e)", row=2, col=1)

# Save as HTML and high-resolution JPEG
bar_html = PLOTS_DIR / "total_costs_ghg_bars.html"
bar_jpeg = PLOTS_DIR / "total_costs_ghg_bars.jpeg"
fig_bars.write_html(str(bar_html))
fig_bars.write_image(str(bar_jpeg), width=1200, height=1000, scale=3)
fig_bars.show()

print("✅ Bar plots created")
print(f"   Saved to: {bar_html}")
print(f"   Saved to: {bar_jpeg}")
print(f"\n📊 Summary Statistics:")
print(f"   Baseline Total Cost: ${total_costs[0]:,.0f}")
print(f"   Baseline Total GHG: {total_ghg[0]:,.0f} metric tons CO2e")
print("\n   Cost Savings vs Baseline:")
for i, scenario in enumerate(scenarios_bar[1:], 1):
    print(f"      {scenario}: {cost_savings_vs_baseline[i]:+.1f}%")

print("\n   GHG Savings vs Baseline:")

for i, scenario in enumerate(scenarios_bar[1:], 1):    print(f"      {scenario}: {ghg_savings_vs_baseline[i]:+.1f}%")


📊 Creating bar plots for total costs and GHG emissions...


✅ Bar plots created
   Saved to: plots/total_costs_ghg_bars.html
   Saved to: plots/total_costs_ghg_bars.jpeg

📊 Summary Statistics:
   Baseline Total Cost: $486,717,579
   Baseline Total GHG: 1,133,335 metric tons CO2e

   Cost Savings vs Baseline:
      NREL_Electric: -6.5%
      NREL_DualFuel: +0.4%
      HPC_Electric: -0.1%
      HPC_DualFuel: +3.6%

   GHG Savings vs Baseline:
      NREL_Electric: +8.5%
      NREL_DualFuel: +7.7%
      HPC_Electric: +14.1%
      HPC_DualFuel: +15.2%


In [12]:
# =============================================================================
# BAR PLOTS: Total Costs and Gas-Only GHG Emissions by Scenario
# =============================================================================

print("\n📊 Creating bar plots for total costs and gas-only GHG emissions...")

# Calculate totals across all stores for each scenario
scenarios_bar_gas = ["Baseline", "NREL_Electric", "NREL_DualFuel", "HPC_Electric", "HPC_DualFuel"]

# Prepare data for costs (same as before)
total_costs_gas = []
total_elec_costs_gas = []
total_gas_costs_gas = []
cost_savings_vs_baseline_gas = []

for scenario in scenarios_bar_gas:
    total_cost_col = f"{scenario}_Total_Energy_Cost"
    elec_consumption_col = f"{scenario}_Electric_Consumption_Cost"
    elec_demand_col = f"{scenario}_Electric_Demand_Cost"
    gas_cost_col = f"{scenario}_Gas_Cost"
    
    total_cost = df_stores_with_costs[total_cost_col].sum()
    elec_cost = (df_stores_with_costs[elec_consumption_col].sum() + 
                 df_stores_with_costs[elec_demand_col].sum())
    gas_cost = df_stores_with_costs[gas_cost_col].sum()
    
    total_costs_gas.append(total_cost)
    total_elec_costs_gas.append(elec_cost)
    total_gas_costs_gas.append(gas_cost)
    
    if scenario == "Baseline":
        cost_savings_vs_baseline_gas.append(0)
    else:
        baseline_total = total_costs_gas[0]
        savings_pct = (baseline_total - total_cost) / baseline_total * 100
        cost_savings_vs_baseline_gas.append(savings_pct)

# Prepare data for GAS-ONLY GHG emissions from city_scenario_emissions
total_gas_ghg_only = []
gas_ghg_savings_vs_baseline = []

# Map scenario names
scenario_to_city_name = {
    "Baseline": "Baseline",
    "NREL_Electric": "2 Speed - Lab Data - Electric Backup",
    "NREL_DualFuel": "2 Speed - Lab Data - Gas Backup",
    "HPC_Electric": "HPC curves - Electric Backup",
    "HPC_DualFuel": "HP Challenge curves - Gas Backup"
}

# Calculate gas GHG for each scenario by scaling from city_scenario_emissions
for scenario in scenarios_bar_gas:
    city_scenario_name = scenario_to_city_name[scenario]
    
    # Aggregate gas GHG across all stores
    total_gas_ghg_kg = 0
    
    for idx, row in df_stores_with_costs[stores_with_energy].iterrows():
        store_cz = str(row['climate zone'])
        
        # Find matching city by climate zone
        matching_city = None
        for city, cz in city_to_cz.items():
            if cz == store_cz:
                matching_city = city
                break
        
        if matching_city is None:
            continue
        
        city_data = city_scenario_emissions[matching_city]
        
        if city_scenario_name in city_data['scenarios']:
            scenario_data = city_data['scenarios'][city_scenario_name]
            gas_ghg_kg_model = scenario_data['gas_ghg_kg']
            
            if gas_ghg_kg_model is not None:
                # Scale by square footage if available
                if 'Square Footage' in df_stores_with_costs.columns:
                    store_sqft = row['Square Footage']
                    if not pd.isna(store_sqft):
                        scaled_gas_ghg = gas_ghg_kg_model * (store_sqft / MODEL_AREA_SQFT)
                    else:
                        scaled_gas_ghg = gas_ghg_kg_model
                else:
                    scaled_gas_ghg = gas_ghg_kg_model
                
                total_gas_ghg_kg += scaled_gas_ghg
    
    # Convert to metric tons
    total_gas_ghg_tons = total_gas_ghg_kg / 1000
    total_gas_ghg_only.append(total_gas_ghg_tons)
    
    if scenario == "Baseline":
        gas_ghg_savings_vs_baseline.append(0)
    else:
        baseline_gas_ghg = total_gas_ghg_only[0]
        if baseline_gas_ghg > 0:
            savings_pct = (baseline_gas_ghg - total_gas_ghg_tons) / baseline_gas_ghg * 100
            gas_ghg_savings_vs_baseline.append(savings_pct)
        else:
            gas_ghg_savings_vs_baseline.append(0)

# Create labels
labels_gas = [scenario_labels_bar[s] for s in scenarios_bar_gas]

# Create figure with 2 subplots
fig_bars_gas = make_subplots(
    rows=2, cols=1,
    subplot_titles=["Total Annual Energy Costs by Scenario", 
                    "Total Annual Gas-Only GHG Emissions by Scenario"],
    vertical_spacing=0.15,
    specs=[[{"type": "bar"}], [{"type": "bar"}]]
)

# Top subplot: Costs (in Million $)
for i, scenario in enumerate(scenarios_bar_gas):
    elec_cost_m = total_elec_costs_gas[i] / 1e6
    gas_cost_m = total_gas_costs_gas[i] / 1e6
    
    # Add electricity cost (bottom part)
    fig_bars_gas.add_trace(
        go.Bar(
            x=[labels_gas[i]],
            y=[elec_cost_m],
            name="Electricity Cost" if i == 0 else "",
            marker_color='#4A90E2',
            showlegend=(i == 0),
            legendgroup="elec",
            text=f"${elec_cost_m:.0f}M",
            textposition='inside',
            textfont=dict(size=12, color='white', family='Arial Black'),
            hovertemplate=f"Electricity: ${total_elec_costs_gas[i]:,.0f}<extra></extra>"
        ),
        row=1, col=1
    )
    
    # Add gas cost (top part)
    fig_bars_gas.add_trace(
        go.Bar(
            x=[labels_gas[i]],
            y=[gas_cost_m],
            name="Gas Cost" if i == 0 else "",
            marker_color='#E67E22',
            showlegend=(i == 0),
            legendgroup="gas",
            text=f"${gas_cost_m:.0f}M",
            textposition='inside',
            textfont=dict(size=12, color='white', family='Arial Black'),
            hovertemplate=f"Gas: ${total_gas_costs_gas[i]:,.0f}<extra></extra>"
        ),
        row=1, col=1
    )
    
    # Add savings percentage (1 decimal, skip for baseline)
    if scenario != "Baseline":
        fig_bars_gas.add_annotation(
            x=labels_gas[i],
            y=elec_cost_m + gas_cost_m,
            text=f"{cost_savings_vs_baseline_gas[i]:+.1f}%",
            showarrow=False,
            yshift=20,
            font=dict(size=11, color='green' if cost_savings_vs_baseline_gas[i] > 0 else 'red'),
            row=1, col=1
        )

# Bottom subplot: Gas-Only GHG Emissions (in k Metric tons)
for i, scenario in enumerate(scenarios_bar_gas):
    gas_ghg_k = total_gas_ghg_only[i] / 1000
    
    # Add Gas GHG bar
    fig_bars_gas.add_trace(
        go.Bar(
            x=[labels_gas[i]],
            y=[gas_ghg_k],
            name="Gas GHG Emissions" if i == 0 else "",
            marker_color='#E67E22',  # Orange to match gas
            showlegend=False,
            text=f"{gas_ghg_k:.0f}k",
            textposition='inside',
            textfont=dict(size=12, color='white', family='Arial Black'),
            hovertemplate=f"Gas GHG: {total_gas_ghg_only[i]:,.0f} tons CO2e<extra></extra>"
        ),
        row=2, col=1
    )
    
    # Add savings percentage (1 decimal, skip for baseline)
    if scenario != "Baseline":
        fig_bars_gas.add_annotation(
            x=labels_gas[i],
            y=gas_ghg_k,
            text=f"{gas_ghg_savings_vs_baseline[i]:+.1f}%",
            showarrow=False,
            yshift=20,
            font=dict(size=11, color='green' if gas_ghg_savings_vs_baseline[i] > 0 else 'red'),
            row=2, col=1
        )

# Update layout
fig_bars_gas.update_layout(
    barmode='stack',
    height=1000,
    width=1200,
    template="plotly_white",
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    ),
    title=dict(
        text="Total Energy Costs and Gas-Only GHG Emissions by Scenario",
        font=dict(size=20),
        x=0.5,
        xanchor="center",
    ),
)

# Update axes
fig_bars_gas.update_yaxes(title_text="Total Cost (Million $)", row=1, col=1)
fig_bars_gas.update_yaxes(title_text="Gas-Only GHG (k metric tons CO2e)", row=2, col=1)

# Save as HTML and high-resolution JPEG
bar_gas_html = PLOTS_DIR / "total_costs_gas_ghg_bars.html"
bar_gas_jpeg = PLOTS_DIR / "total_costs_gas_ghg_bars.jpeg"
fig_bars_gas.write_html(str(bar_gas_html))
fig_bars_gas.write_image(str(bar_gas_jpeg), width=1200, height=1000, scale=3)
fig_bars_gas.show()

print("✅ Gas-only GHG bar plots created")
print(f"   Saved to: {bar_gas_html}")
print(f"   Saved to: {bar_gas_jpeg}")
print(f"\n📊 Gas-Only GHG Summary Statistics:")
print(f"   Baseline Gas GHG: {total_gas_ghg_only[0]:,.0f} metric tons CO2e")

print("\n   Gas GHG Savings vs Baseline:")

for i, scenario in enumerate(scenarios_bar_gas[1:], 1):    print(f"      {scenario}: {gas_ghg_savings_vs_baseline[i]:+.1f}%")


📊 Creating bar plots for total costs and gas-only GHG emissions...


✅ Gas-only GHG bar plots created
   Saved to: plots/total_costs_gas_ghg_bars.html
   Saved to: plots/total_costs_gas_ghg_bars.jpeg

📊 Gas-Only GHG Summary Statistics:
   Baseline Gas GHG: 273,531 metric tons CO2e

   Gas GHG Savings vs Baseline:
      NREL_Electric: +100.0%
      NREL_DualFuel: +51.2%
      HPC_Electric: +100.0%
      HPC_DualFuel: +89.1%


In [13]:
# =============================================================================
# DELTA SAVINGS MAP PLOTS: Electric vs Dual Fuel Comparison (GHG)
# =============================================================================

print("\n🗺️ Creating delta GHG savings maps (Electric vs Dual Fuel)...")

# Calculate delta GHG savings by state for both NREL and HPC scenarios
# Delta = Electric savings - Dual Fuel savings (positive means electric saves more GHG)

# Approximate state center coordinates (latitude, longitude)
state_coords = {
    'AL': (32.806671, -86.791130), 'AK': (61.370716, -152.404419),
    'AZ': (33.729759, -111.431221), 'AR': (34.969704, -92.373123),
    'CA': (36.116203, -119.681564), 'CO': (39.059811, -105.311104),
    'CT': (41.597782, -72.755371), 'DE': (39.318523, -75.507141),
    'FL': (27.766279, -81.686783), 'GA': (33.040619, -83.643074),
    'HI': (21.094318, -157.498337), 'ID': (44.240459, -114.478828),
    'IL': (40.349457, -88.986137), 'IN': (39.849426, -86.258278),
    'IA': (42.011539, -93.210526), 'KS': (38.526600, -96.726486),
    'KY': (37.668140, -84.670067), 'LA': (31.169546, -91.867805),
    'ME': (44.693947, -69.381927), 'MD': (39.063946, -76.802101),
    'MA': (42.230171, -71.530106), 'MI': (43.326618, -84.536095),
    'MN': (45.694454, -93.900192), 'MS': (32.741646, -89.678696),
    'MO': (38.456085, -92.288368), 'MT': (46.921925, -110.454353),
    'NE': (41.125370, -98.268082), 'NV': (38.313515, -117.055374),
    'NH': (43.452492, -71.563896), 'NJ': (40.298904, -74.521011),
    'NM': (34.840515, -106.248482), 'NY': (42.165726, -74.948051),
    'NC': (35.630066, -79.806419), 'ND': (47.528912, -99.784012),
    'OH': (40.388783, -82.764915), 'OK': (35.565342, -96.928917),
    'OR': (44.572021, -122.070938), 'PA': (40.590752, -77.209755),
    'RI': (41.680893, -71.511780), 'SC': (33.856892, -80.945007),
    'SD': (44.299782, -99.438828), 'TN': (35.747845, -86.692345),
    'TX': (31.054487, -97.563461), 'UT': (40.150032, -111.862434),
    'VT': (44.045876, -72.710686), 'VA': (37.769337, -78.169968),
    'WA': (47.400902, -121.490494), 'WV': (38.491226, -80.954453),
    'WI': (44.268543, -89.616508), 'WY': (42.755966, -107.302490)
}

# Scenarios to compare
comparison_scenarios = [
    {
        'name': 'NREL Lab Data',
        'electric': 'NREL_Electric',
        'dual_fuel': 'NREL_DualFuel',
        'title': 'NREL Lab Data:<br>Electric vs Dual Fuel GHG Savings Delta'
    },
    {
        'name': 'HP Challenge',
        'electric': 'HPC_Electric',
        'dual_fuel': 'HPC_DualFuel',
        'title': 'HP Challenge:<br>Electric vs Dual Fuel GHG Savings Delta'
    }
]

# Create figure with 1 row, 2 columns
fig_delta = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "choropleth"}, {"type": "choropleth"}]],
    subplot_titles=[comp['title'] for comp in comparison_scenarios],
    horizontal_spacing=0.01,
    vertical_spacing=0.15
)

all_delta_values = []

for idx, comp in enumerate(comparison_scenarios):
    col = idx + 1
    
    # Get GHG savings columns for electric and dual fuel
    elec_savings_col = f"{comp['electric']}_GHG_Savings_vs_Baseline_Percent"
    dual_savings_col = f"{comp['dual_fuel']}_GHG_Savings_vs_Baseline_Percent"
    
    # Calculate delta by state (mean)
    state_delta = {}
    
    for state in df_stores_with_costs['State'].unique():
        if pd.isna(state):
            continue
            
        state_data = df_stores_with_costs[df_stores_with_costs['State'] == state]
        
        # Calculate mean GHG savings for each scenario
        elec_mean = state_data[elec_savings_col].mean()
        dual_mean = state_data[dual_savings_col].mean()
        
        # Skip states with no data (NaN values)
        if pd.isna(elec_mean) or pd.isna(dual_mean):
            continue
        
        # Delta = electric - dual fuel
        delta = elec_mean - dual_mean
        
        state_delta[state] = delta
        all_delta_values.append(delta)
    
    # Prepare data for choropleth
    states = list(state_delta.keys())
    deltas = list(state_delta.values())
    
    # Create choropleth trace with red-green colorscale
    fig_delta.add_trace(
        go.Choropleth(
            locations=states,
            z=deltas,
            locationmode='USA-states',
            colorscale=[[0, 'rgb(215, 25, 28)'],    # Red (negative - dual fuel better)
                        [0.5, 'rgb(255, 255, 191)'], # Light yellow (neutral)
                        [1, 'rgb(26, 150, 65)']],    # Green (positive - electric better)
            zmid=0,  # Center colorscale at 0
            colorbar=dict(
                title="Delta (%)",
                len=0.9,
                y=0.5,
            ) if col == 2 else dict(showticklabels=False, len=0.9, y=0.5),
            marker_line_color='white',
            marker_line_width=1,
            hovertemplate='<b>%{location}</b><br>Delta: %{z:.1f}%<extra></extra>',
        ),
        row=1, col=col
    )
    
    # Add state labels with delta values on top of each state
    for state, delta in state_delta.items():
        if state in state_coords:
            lat, lon = state_coords[state]
            fig_delta.add_scattergeo(
                lon=[lon],
                lat=[lat],
                text=[f"{delta:.1f}"],
                mode='text',
                textfont=dict(size=11, color='black', family='Arial Black'),
                showlegend=False,
                row=1, col=col,
            )

# Calculate symmetric range for consistent coloring
if all_delta_values:
    max_abs_delta = max(abs(min(all_delta_values)), abs(max(all_delta_values)))
else:
    max_abs_delta = 1

# Update layout
fig_delta.update_geos(
    scope='usa',
    projection_type='albers usa',
    showlakes=False,
    showcountries=False,
    showcoastlines=False,
    showland=True,
    landcolor='rgb(243, 243, 243)',
)

fig_delta.update_layout(
    title=dict(
        text="GHG Savings Delta: Electric vs Dual Fuel by State<br><sub>Delta = (Electric GHG Savings) - (Dual Fuel GHG Savings)<br>Positive values: Electric saves more GHG | Negative values: Dual Fuel saves more GHG</sub>",
        font=dict(size=18),
        x=0.5,
        xanchor='center',
        y=0.98,
        yanchor='top'
    ),
    height=650,
    width=1600,
    template="plotly_white",
    showlegend=False,
    margin=dict(t=150, b=50, l=50, r=50),
)

# Update subplot title positions to avoid overlap
for annotation in fig_delta.layout.annotations:
    annotation.update(y=annotation.y - 0.03)  # Move subplot titles down slightly

# Save as HTML and high-resolution JPEG
delta_html = PLOTS_DIR / "electric_vs_dual_fuel_ghg_delta.html"
delta_jpeg = PLOTS_DIR / "electric_vs_dual_fuel_ghg_delta.jpeg"
fig_delta.write_html(str(delta_html))
fig_delta.write_image(str(delta_jpeg), width=1600, height=650, scale=3)
fig_delta.show()

print("✅ Delta GHG savings maps created")
print(f"   Saved to: {delta_html}")
print(f"   Saved to: {delta_jpeg}")
print(f"\n📊 Delta Summary (GHG Savings):")
print("   Formula: Delta = (Electric GHG Savings) - (Dual Fuel GHG Savings)")
for comp in comparison_scenarios:
    elec_savings_col = f"{comp['electric']}_GHG_Savings_vs_Baseline_Percent"
    dual_savings_col = f"{comp['dual_fuel']}_GHG_Savings_vs_Baseline_Percent"
    
    elec_mean = df_stores_with_costs[elec_savings_col].mean()
    dual_mean = df_stores_with_costs[dual_savings_col].mean()
    delta = elec_mean - dual_mean
    
    print(f"\n   {comp['name']}:")
    print(f"      Electric Mean GHG Savings: {elec_mean:.1f}%")
    print(f"      Dual Fuel Mean GHG Savings: {dual_mean:.1f}%")
    print(f"      Delta (Electric - Dual Fuel): {delta:.1f}%")

# Check for states with missing GHG data
print(f"\n⚠️  States with no GHG data (excluded from maps):")
all_states = set(df_stores_with_costs['State'].dropna().unique())
states_with_data = set()
for comp in comparison_scenarios:
    elec_col = f"{comp['electric']}_GHG_Savings_vs_Baseline_Percent"
    for state in all_states:
        state_data = df_stores_with_costs[df_stores_with_costs['State'] == state]
        if state_data[elec_col].notna().any():
            states_with_data.add(state)
            break

missing_states = sorted(all_states - states_with_data)
if missing_states:
    print(f"   {', '.join(missing_states)}")
else:
    print("   None - all states have data")


🗺️ Creating delta GHG savings maps (Electric vs Dual Fuel)...


✅ Delta GHG savings maps created
   Saved to: plots/electric_vs_dual_fuel_ghg_delta.html
   Saved to: plots/electric_vs_dual_fuel_ghg_delta.jpeg

📊 Delta Summary (GHG Savings):
   Formula: Delta = (Electric GHG Savings) - (Dual Fuel GHG Savings)

   NREL Lab Data:
      Electric Mean GHG Savings: 11.1%
      Dual Fuel Mean GHG Savings: 9.8%
      Delta (Electric - Dual Fuel): 1.4%

   HP Challenge:
      Electric Mean GHG Savings: 16.3%
      Dual Fuel Mean GHG Savings: 16.9%
      Delta (Electric - Dual Fuel): -0.5%

⚠️  States with no GHG data (excluded from maps):
   AK, AL, AR, AZ, CA, CO, CT, DC, DE, FL, HI, IA, ID, IL, IN, KS, KY, LA, MA, MD, ME, MI, MN, MO, MS, MT, NC, ND, NE, NH, NJ, NM, NV, NY, OH, OK, OR, PA, RI, SC, SD, TN, TX, UT, VA, VT, WA, WI, WV, WY


In [14]:
# =============================================================================
# STATE-BY-STATE SUMMARY: produce detailed CSVs per scenario and pivot tables
# =============================================================================
from pathlib import Path
import pandas as pd
import numpy as np

PLOTS_DIR = Path('plots')
PLOTS_DIR.mkdir(exist_ok=True)

# Scenarios to summarize (consistent with plotting code)
scenarios = ['Baseline','NREL_Electric','NREL_DualFuel','HPC_Electric','HPC_DualFuel']

if 'df_stores_with_costs' not in globals():
    raise RuntimeError('df_stores_with_costs not found in notebook globals — run analysis cells first')

df = df_stores_with_costs
states = sorted(df['State'].dropna().unique())
rows = []
cols = set(df.columns.tolist())

for state in states:
    state_df = df[df['State'] == state]
    n_state = len(state_df)
    for s in scenarios:
        cost_col = f'{s}_Total_Energy_Cost'
        mean_cost_col = None
        ghg_col = f'{s}_GHG_Emissions_kg'
        savings_cost_col = f'{s}_Savings_vs_Baseline_Percent'
        ghg_savings_col = f'{s}_GHG_Savings_vs_Baseline_Percent'

        n_with_cost = int(state_df[cost_col].notna().sum()) if cost_col in cols else 0
        total_cost = float(state_df[cost_col].sum(skipna=True)) if (cost_col in cols and n_with_cost>0) else np.nan
        mean_cost = float(state_df[cost_col].dropna().mean()) if (cost_col in cols and n_with_cost>0) else np.nan

        n_with_ghg = int(state_df[ghg_col].notna().sum()) if ghg_col in cols else 0
        total_ghg_tons = float(state_df[ghg_col].sum(skipna=True))/1000.0 if (ghg_col in cols and n_with_ghg>0) else np.nan
        mean_ghg_tons = float(state_df[ghg_col].dropna().mean())/1000.0 if (ghg_col in cols and n_with_ghg>0) else np.nan

        mean_cost_savings_pct = float(state_df[savings_cost_col].dropna().mean()) if savings_cost_col in cols and state_df[savings_cost_col].notna().any() else np.nan
        mean_ghg_savings_pct = float(state_df[ghg_savings_col].dropna().mean()) if ghg_savings_col in cols and state_df[ghg_savings_col].notna().any() else np.nan

        rows.append({
            'state': state,
            'n_stores_in_state': n_state,
            'scenario': s,
            'n_with_cost': n_with_cost,
            'total_cost_usd': total_cost,
            'mean_cost_usd_per_store': mean_cost,
            'n_with_ghg': n_with_ghg,
            'total_ghg_tons': total_ghg_tons,
            'mean_ghg_tons_per_store': mean_ghg_tons,
            'mean_cost_savings_pct_vs_baseline': mean_cost_savings_pct,
            'mean_ghg_savings_pct_vs_baseline': mean_ghg_savings_pct
        })

# Long form CSV (one row per state x scenario)
df_state_scenario = pd.DataFrame(rows)
out_long = PLOTS_DIR / 'state_scenario_summary.csv'
df_state_scenario.to_csv(out_long, index=False)
print(f'✅ Wrote long-form state x scenario summary to: {out_long}')

# Create pivot tables for convenient comparison (states as rows, scenarios as columns)
pivots = {}
metrics = {
    'total_cost_usd':'total_cost_usd',
    'mean_cost_usd_per_store':'mean_cost_usd_per_store',
    'total_ghg_tons':'total_ghg_tons',
    'mean_ghg_tons_per_store':'mean_ghg_tons_per_store',
    'mean_cost_savings_pct_vs_baseline':'mean_cost_savings_pct_vs_baseline',
    'mean_ghg_savings_pct_vs_baseline':'mean_ghg_savings_pct_vs_baseline',
    'n_with_cost':'n_with_cost',
    'n_with_ghg':'n_with_ghg'
}
for key, col in metrics.items():
    try:
        pivot = df_state_scenario.pivot(index='state', columns='scenario', values=col)
        outp = PLOTS_DIR / f'state_pivot_{key}.csv'
        pivot.to_csv(outp)
        pivots[key] = outp
        print(f'✅ Wrote pivot for {key} to: {outp}')
    except Exception as e:
        print(f'⚠️ Could not create pivot for {key}: {e}')

# Save a compact per-state summary (aggregate across scenarios) for quick reference
agg_rows = []
for state in df_state_scenario['state'].unique():
    srows = df_state_scenario[df_state_scenario['state'] == state]
    agg_rows.append({
        'state': state,
        'total_stores': int(srows['n_stores_in_state'].iloc[0]) if not srows.empty else 0,
        'scenarios_available': ','.join(sorted(srows['scenario'].dropna().unique())),
        'total_cost_all_scenarios_usd': float(srows['total_cost_usd'].fillna(0).sum()),
        'total_ghg_all_scenarios_tons': float(srows['total_ghg_tons'].fillna(0).sum()),
    })
df_state_agg = pd.DataFrame(agg_rows).sort_values('state')
out_agg = PLOTS_DIR / 'state_summary_aggregate.csv'
df_state_agg.to_csv(out_agg, index=False)
print(f'✅ Wrote compact per-state aggregate to: {out_agg}')

# Show sample of the long form table
display(df_state_scenario.head(10))

# Return the long dataframe so notebook shows it as output
df_state_scenario.to_csv('Summary_17F.csv')
df_state_scenario


✅ Wrote long-form state x scenario summary to: plots/state_scenario_summary.csv
✅ Wrote pivot for total_cost_usd to: plots/state_pivot_total_cost_usd.csv
✅ Wrote pivot for mean_cost_usd_per_store to: plots/state_pivot_mean_cost_usd_per_store.csv
✅ Wrote pivot for total_ghg_tons to: plots/state_pivot_total_ghg_tons.csv
✅ Wrote pivot for mean_ghg_tons_per_store to: plots/state_pivot_mean_ghg_tons_per_store.csv
✅ Wrote pivot for mean_cost_savings_pct_vs_baseline to: plots/state_pivot_mean_cost_savings_pct_vs_baseline.csv
✅ Wrote pivot for mean_ghg_savings_pct_vs_baseline to: plots/state_pivot_mean_ghg_savings_pct_vs_baseline.csv
✅ Wrote pivot for n_with_cost to: plots/state_pivot_n_with_cost.csv
✅ Wrote pivot for n_with_ghg to: plots/state_pivot_n_with_ghg.csv
✅ Wrote compact per-state aggregate to: plots/state_summary_aggregate.csv


,state,n_stores_in_state,scenario,n_with_cost,total_cost_usd,mean_cost_usd_per_store,n_with_ghg,total_ghg_tons,mean_ghg_tons_per_store,mean_cost_savings_pct_vs_baseline,mean_ghg_savings_pct_vs_baseline
0,AK,3,Baseline,3,1.029177e+06,343059.029788,3,2087.686023,695.895341,NaN,NaN
1,AK,3,NREL_Electric,3,1.348289e+06,449429.599827,3,1773.227384,591.075795,-31.006492,15.062545
2,AK,3,NREL_DualFuel,3,1.076754e+06,358918.051422,3,1909.578760,636.526253,-4.622826,8.531324
3,AK,3,HPC_Electric,3,1.232180e+06,410726.676158,3,1595.966673,531.988891,-19.724782,23.553319
4,AK,3,HPC_DualFuel,3,1.059778e+06,353259.197657,3,1581.209746,527.069915,-2.973298,24.260175
5,AL,23,Baseline,23,5.046033e+06,219392.739932,23,14441.669490,627.898673,NaN,NaN
6,AL,23,NREL_Electric,23,4.983697e+06,216682.469232,23,13913.196899,604.921604,1.237235,5.072554
7,AL,23,NREL_DualFuel,23,4.919245e+06,213880.228234,23,13836.840586,601.601765,2.523314,5.427929
8,AL,23,HPC_Electric,23,4.721263e+06,205272.299299,23,13251.545086,576.154134,6.435181,9.686323
9,AL,23,HPC_DualFuel,23,4.683021e+06,203609.606125,23,13163.254622,572.315418,7.197673,10.138310


,state,n_stores_in_state,scenario,n_with_cost,total_cost_usd,mean_cost_usd_per_store,n_with_ghg,total_ghg_tons,mean_ghg_tons_per_store,mean_cost_savings_pct_vs_baseline,mean_ghg_savings_pct_vs_baseline
0,AK,3,Baseline,3,1.029177e+06,343059.029788,3,2087.686023,695.895341,NaN,NaN
1,AK,3,NREL_Electric,3,1.348289e+06,449429.599827,3,1773.227384,591.075795,-31.006492,15.062545
2,AK,3,NREL_DualFuel,3,1.076754e+06,358918.051422,3,1909.578760,636.526253,-4.622826,8.531324
3,AK,3,HPC_Electric,3,1.232180e+06,410726.676158,3,1595.966673,531.988891,-19.724782,23.553319
4,AK,3,HPC_DualFuel,3,1.059778e+06,353259.197657,3,1581.209746,527.069915,-2.973298,24.260175
...,...,...,...,...,...,...,...,...,...,...,...
250,WY,3,Baseline,3,5.347342e+05,178244.737740,3,1504.490561,501.496854,NaN,NaN
251,WY,3,NREL_Electric,3,6.884125e+05,229470.839696,3,1047.356193,349.118731,-28.596538,34.068330
252,WY,3,NREL_DualFuel,3,5.431176e+05,181039.193254,3,1269.810278,423.270093,-1.538412,17.297747
253,WY,3,HPC_Electric,3,6.370877e+05,212362.566014,3,949.113825,316.371275,-18.997347,40.126859
